In [ ]:
# =========================================================
# CELL 1 — Imports + Reproducibility
# Purpose:
#   - Import all dependencies
#   - Fix random seeds for reproducibility
# Inputs:  None
# Outputs: Global seeded RNG state
# =========================================================

import os
import re
import math
import ast
import json
import pickle
import random
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# NEW CELL 1.5 — Device (define early so later cells can use it)
# =========================================================
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# =========================================================
# CELL 2 — Load CSVs + Embeddings
# Purpose:
#   - Load all datasets
#   - Load all embedding dicts (id -> np.array)
# Inputs:
#   - Paths to CSVs and .pkl embeddings
# Outputs:
#   - main_df, dialog_df, docs_df, resp_df, summ_df
#   - emb_dialog, emb_docs, emb_resp, emb_summ
# =========================================================

MAIN_PATH   = r"/teamspace/studios/this_studio/datasets/main_mixed_df.csv"
DIALOG_PATH = r"/teamspace/studios/this_studio/datasets/dialogue.csv"
DOCS_PATH   = r"/teamspace/studios/this_studio/datasets/documents.csv"
RESP_PATH   = r"/teamspace/studios/this_studio/datasets/response.csv"
SUMM_PATH   = r"/teamspace/studios/this_studio/datasets/summaries.csv"

EMB_DIALOG_PATH = r"/teamspace/studios/this_studio/embeddings/dialog_embeddings.pkl"
EMB_DOCS_PATH   = r"/teamspace/studios/this_studio/embeddings/doc_embeddings.pkl"
EMB_RESP_PATH   = r"/teamspace/studios/this_studio/embeddings/response_embeddings.pkl"
EMB_SUMM_PATH   = r"/teamspace/studios/this_studio/embeddings/summ_embeddings.pkl"

main_df  = pd.read_csv(MAIN_PATH)
dialog_df= pd.read_csv(DIALOG_PATH)
docs_df  = pd.read_csv(DOCS_PATH)
resp_df  = pd.read_csv(RESP_PATH)
summ_df  = pd.read_csv(SUMM_PATH)

def load_embeddings(file_path):
    with open(file_path, "rb") as f:
        return pickle.load(f)

emb_dialog = load_embeddings(EMB_DIALOG_PATH)
emb_docs   = load_embeddings(EMB_DOCS_PATH)
emb_resp   = load_embeddings(EMB_RESP_PATH)
emb_summ   = load_embeddings(EMB_SUMM_PATH)

print("Loaded main_df columns:", list(main_df.columns))
print("Rows:", len(main_df))
print("Sample row:\n", main_df.iloc[0])

In [3]:
# =========================================================
# CELL 3 — Core Helpers (sequence parse, canonical ids, null checks)
# Purpose:
#   - Robust parsing of serialized sequences
#   - Canonical ID utilities
# Inputs:  main_df cell strings
# Outputs: Helper functions used everywhere
# =========================================================

CANON_RE = re.compile(r"^(dl|d|r|s)-\d+$", re.IGNORECASE)

def is_nullish(x):
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    s = str(x).strip().lower()
    return s in {"", "null", "none", "nan"}

def normalize_id(x):
    return str(x).strip()

def to_canonical(prefix: str, raw_id):
    s = str(raw_id).strip()
    if CANON_RE.match(s):
        return s
    return f"{prefix}-{s}"

def parse_serialized_sequence(x):
    """
    Accepts:
      - Python list
      - '["a","b"]'
      - '[a,b]'
    Returns: list[str]
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip() != ""]
    s = str(x).strip()
    if not (s.startswith("[") and s.endswith("]")):
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return [str(t).strip() for t in v if str(t).strip() != ""]
    except Exception:
        pass
    items = [t.strip().strip('"').strip("'") for t in s[1:-1].split(",")]
    return [t for t in items if t]

def fix_and_align_sequences(nodes, actions, max_len=100):
    nodes = list(nodes)
    actions = list(actions)
    L = min(len(nodes), len(actions), max_len)
    return nodes[:L], actions[:L]

In [ ]:
# =========================================================
# CELL 4 — Build id2text (canonical) from all text-bearing tables
# Purpose:
#   - Create mapping canonical_id -> text
# Inputs:
#   - dialog_df, docs_df, resp_df, summ_df
# Outputs:
#   - id2text dict
# =========================================================

def build_id2text(dialog_df, docs_df, resp_df, summ_df):
    id2text = {}

    # Dialogue: dl-*
    if "dialogue_id" in dialog_df.columns:
        for _, r in dialog_df.dropna(subset=["dialogue_id"]).iterrows():
            cid = to_canonical("dl", r["dialogue_id"])
            id2text[cid] = str(r.get("dialogue_text", "")).strip()

    # Docs: d-*
    if "NewsID" in docs_df.columns:
        for _, r in docs_df.dropna(subset=["NewsID"]).iterrows():
            cid = to_canonical("d", r["NewsID"])
            nb = str(r.get("NewsBody", "")).strip()
            hl = str(r.get("Headline", "")).strip()
            id2text[cid] = nb if nb else hl

    # Responses: r-*
    if "response_id" in resp_df.columns:
        for _, r in resp_df.dropna(subset=["response_id"]).iterrows():
            cid = to_canonical("r", r["response_id"])
            id2text[cid] = str(r.get("response_text", "")).strip()

    # Summaries: s-*
    if "SummID" in summ_df.columns:
        for _, r in summ_df.dropna(subset=["SummID"]).iterrows():
            cid = to_canonical("s", r["SummID"])
            id2text[cid] = str(r.get("Summary", "")).strip()

    return id2text

id2text = build_id2text(dialog_df, docs_df, resp_df, summ_df)
print("id2text size:", len(id2text))
print("sample keys:", list(id2text.keys())[:10])


In [ ]:
# =========================================================
# CELL 5 — Embedding Resolver for canonical ids
# Purpose:
#   - Provide emb_resolver.get(cid) -> np.array(768,)
# Inputs:
#   - emb_dialog, emb_docs, emb_resp, emb_summ
# Outputs:
#   - emb_resolver
# =========================================================

class EmbLookup:
    @staticmethod
    def lookup(emb_dict, key):
        if key in emb_dict:
            return emb_dict[key]
        s = str(key)
        if s in emb_dict:
            return emb_dict[s]
        nums = re.findall(r"\d+", s)
        if nums:
            n = nums[-1]
            if n in emb_dict:
                return emb_dict[n]
            try:
                ni = int(n)
                if ni in emb_dict:
                    return emb_dict[ni]
            except Exception:
                pass
        return None

class CanonicalEmbeddingResolver:
    def __init__(self, emb_dialog, emb_docs, emb_resp, emb_summ, d=768):
        self.emb_dialog = emb_dialog
        self.emb_docs = emb_docs
        self.emb_resp = emb_resp
        self.emb_summ = emb_summ
        self.d = d

    def get(self, cid: str) -> np.ndarray:
        cid = normalize_id(cid)
        prefix = cid.split("-")[0].lower()

        if prefix == "dl":
            v = EmbLookup.lookup(self.emb_dialog, cid)
        elif prefix == "d":
            v = EmbLookup.lookup(self.emb_docs, cid)
        elif prefix == "r":
            v = EmbLookup.lookup(self.emb_resp, cid)
        elif prefix == "s":
            v = EmbLookup.lookup(self.emb_summ, cid)
        else:
            v = None

        if v is None:
            return np.zeros((self.d,), dtype=np.float32)

        v = np.asarray(v, dtype=np.float32)
        if v.shape[-1] != self.d:
            raise ValueError(f"Embedding dim mismatch for {cid}: got {v.shape}, expected ({self.d},)")
        return v

emb_resolver = CanonicalEmbeddingResolver(emb_dialog, emb_docs, emb_resp, emb_summ, d=768)
print("Embedding resolver ready.")


In [ ]:
# =========================================================
# CELL 6 — Task Names + Task IDs
# Purpose:
#   - Build task vocab for task-classification head (all tasks)
# Inputs:
#   - main_df['task']
# Outputs:
#   - task2idx, idx2task, main_df['task_id'], n_tasks
# =========================================================

assert "task" in main_df.columns, "main_df must contain 'task'"

task_names = sorted(main_df["task"].dropna().astype(str).unique())
task2idx = {t: i for i, t in enumerate(task_names)}
idx2task = {i: t for t, i in task2idx.items()}

main_df["task_id"] = main_df["task"].astype(str).map(task2idx)
assert main_df["task_id"].isna().sum() == 0, "Unmapped task found"

n_tasks = len(task2idx)
print("Tasks:", task2idx)


In [ ]:
# =========================================================
# CELL 7 — Build ACTION vocab (all rows) and NODE vocab (all rows)
# Purpose:
#   - actions2idx for sequence action loss
#   - node2idx for sequence node loss
# Inputs:
#   - main_df['interactions'], main_df['text_nodes']
# Outputs:
#   - actions2idx/idx2action
#   - node2idx/idx2node
# =========================================================

# ACTIONS
all_actions = set()
for seq in main_df["interactions"].dropna():
    all_actions.update(parse_serialized_sequence(seq))

actions2idx = {a: i for i, a in enumerate(sorted(all_actions))}
idx2action  = {i: a for a, i in actions2idx.items()}

# NODES (sequence tokens)
node_ids = set()
for seq in main_df["text_nodes"].dropna():
    for nid in parse_serialized_sequence(seq):
        if not is_nullish(nid):
            node_ids.add(normalize_id(nid))

# Include query/target ids if present (safer for lookups)
for col in ["query_text_id", "target_text_id"]:
    if col in main_df.columns:
        for x in main_df[col].dropna():
            if not is_nullish(x):
                node_ids.add(normalize_id(x))

node2idx = {nid: i for i, nid in enumerate(sorted(node_ids))}
idx2node = {i: nid for nid, i in node2idx.items()}

print("VOCAB SIZES:")
print("  actions:", len(actions2idx))
print("  nodes  :", len(node2idx))


In [ ]:
# =========================================================
# CELL 8 — Build USER vocab ONLY from next_user_prediction rows
# Purpose:
#   - user vocabulary restricted to next_user_prediction task
# Inputs:
#   - main_df rows where task == 'next_user_prediction'
# Outputs:
#   - user2idx_small / idx2user_small
# Notes:
#   - For non-user-pred tasks, user loss will be masked out
# =========================================================

USER_TASK = "next_user_prediction"

user_rows = main_df[main_df["task"].astype(str).str.strip() == USER_TASK]
unique_users_small = sorted(user_rows["userid"].dropna().astype(str).unique())

# ensure at least an <unk> for safety
unique_users_small = ["<unk_user>"] + [u for u in unique_users_small if u != "<unk_user>"]

user2idx_small = {u: i for i, u in enumerate(unique_users_small)}
idx2user_small = {i: u for u, i in user2idx_small.items()}

print("User vocab size (next_user_prediction only):", len(user2idx_small))


In [ ]:
# =========================================================
# CELL 9 — Build ITEM vocab ONLY from next_item_prediction rows (query_text_id only)
# Purpose:
#   - item vocabulary restricted to next_item_prediction task
# Inputs:
#   - main_df rows where task == 'next_item_prediction'
# Outputs:
#   - item2idx / idx2item
# Notes:
#   - Vocab is built ONLY from query_text_id by your requirement
#   - Targets (target_text_id) might be OOV; handled via <unk_item>
# =========================================================

ITEM_TASK = "next_item_prediction"

item_rows = main_df[main_df["task"].astype(str).str.strip() == ITEM_TASK]
q_items = item_rows["query_text_id"].dropna().astype(str).map(lambda x: normalize_id(x)).unique().tolist()

q_items = ["<unk_item>"] + [x for x in sorted(set(q_items)) if x != "<unk_item>"]

item2idx = {it: i for i, it in enumerate(q_items)}
idx2item = {i: it for it, i in item2idx.items()}

print("Item vocab size (next_item_prediction query_text_id only):", len(item2idx))


In [ ]:
# =========================================================
# CELL 10 — Build word supervision vocab ONLY for dialogue_generation + personalized_summarization
# Purpose:
#   - Replace YAKE phrases with simple word lists
#   - Build id2words ONLY using query_text_id + target_text_id columns
#     and ONLY for tasks: dialogue_generation, personalized_summarization
# Inputs:
#   - main_df filtered tasks
#   - id2text
# Outputs:
#   - id2words: canonical_id -> list[str]
#   - word2idx/idx2word
# =========================================================

GEN_TASKS_WORD = {"dialogue_generation", "personalized_summarization"}

WORD_MAX_PER_TEXT = 80
WORD_MIN_LEN = 2

WORD_RE = re.compile(r"[a-z0-9]+")  # simple tokenization

def tokenize_words(text: str):
    text = (text or "").lower()
    toks = WORD_RE.findall(text)
    toks = [t for t in toks if len(t) >= WORD_MIN_LEN]
    return toks

# collect unique ids only from query/target cols for these tasks
gen_df = main_df[main_df["task"].astype(str).str.strip().isin(GEN_TASKS_WORD)].copy()

need_cols = ["query_text_id", "target_text_id"]
for c in need_cols:
    assert c in gen_df.columns, f"Missing column {c} for word supervision."

unique_text_ids = set()
for c in need_cols:
    unique_text_ids.update(gen_df[c].dropna().astype(str).map(normalize_id).tolist())

# id2words (only for ids we can resolve to text)
id2words = {}
word_vocab = set()

for tid in unique_text_ids:
    if tid not in id2text:
        continue
    words = tokenize_words(id2text[tid])[:WORD_MAX_PER_TEXT]
    id2words[tid] = words
    word_vocab.update(words)

# vocab with <pad>/<unk> for safety (multi-label does not need pad, but keep unk)
word_vocab = ["<unk_word>"] + [w for w in sorted(word_vocab) if w != "<unk_word>"]
word2idx = {w: i for i, w in enumerate(word_vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print("Texts for word supervision:", len(id2words))
print("Word vocab size:", len(word2idx))

In [ ]:
# =========================================================
# Build per-word importance weights (with progress bar)
# =========================================================

import spacy
from tqdm import tqdm

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # faster
CONTENT_POS = {"NOUN", "PROPN", "ADJ"}

word_importance = torch.ones(len(word2idx), dtype=torch.float32)

# We iterate in vocab order for stability
words = list(word2idx.keys())

for w in tqdm(words, desc="Computing word importance"):
    i = word2idx[w]
    doc = nlp(w)
    if len(doc) > 0 and doc[0].pos_ in CONTENT_POS:
        word_importance[i] = 10.0     # important content words
    else:
        word_importance[i] = 0.02     # function / glue words

word_importance = word_importance.to(device)

print("Word importance ready:", word_importance.shape)


In [ ]:
# =========================================================
# CELL 11 — Split train/test from provided 'split' column (no dropping)
# Purpose:
#   - Use main_df['split'] == train/test
# Inputs:
#   - main_df['split']
# Outputs:
#   - train_df, test_df
# =========================================================

assert "split" in main_df.columns, "main_df must contain 'split' with train/test."

main_df["split"] = main_df["split"].astype(str).str.strip().str.lower()
train_df = main_df[main_df["split"] == "train"].reset_index(drop=True)
test_df  = main_df[main_df["split"] == "test"].reset_index(drop=True)

print("Train size:", len(train_df), "| Test size:", len(test_df))
print("Task counts (train):\n", train_df["task"].value_counts())
print("Task counts (test):\n", test_df["task"].value_counts())

In [13]:
# =========================================================
# CELL 12 — Batch Builder (row -> tensors) with TASK-WISE masks
# (DROP-IN REPLACEMENT: adds q_text/t_text for LM)
# =========================================================

MAX_T = 60
D_MODEL = 768
IGNORE_INDEX = -100

def build_multi_hot_words(text_id: str, device):
    """
    Multi-hot vector over word2idx for the given canonical text_id.
    """
    y = torch.zeros(len(word2idx), device=device)
    words = id2words.get(text_id, [])
    for w in words:
        j = word2idx[w] if w in word2idx else 0
        y[j] = 1.0
    return y, words

def safe_user_label(userid: str):
    userid = str(userid).strip()
    if userid in user2idx_small:
        return user2idx_small[userid]
    return 0

def safe_item_label(itemid: str):
    itemid = normalize_id(itemid)
    if itemid in item2idx:
        return item2idx[itemid]
    return 0

def build_row_tensors(row, device):
    task = str(row["task"]).strip()
    task_id = int(row["task_id"])

    # ---- sequences ----
    actions = parse_serialized_sequence(row["interactions"])
    nodes   = parse_serialized_sequence(row["text_nodes"])

    if len(actions) == 0 or len(nodes) == 0:
        return {"ok": False, "reason": "empty_sequence", "task": task}

    nodes, actions = fix_and_align_sequences(nodes, actions, max_len=MAX_T)
    if len(nodes) < 3:
        return {"ok": False, "reason": "too_short_sequence", "task": task}

    T = len(nodes)

    # ---- action tensors ----
    try:
        action_ids = [actions2idx[a] for a in actions]
    except KeyError:
        return {"ok": False, "reason": "unknown_action_token", "task": task}

    action_labels_full = torch.tensor(action_ids, device=device).unsqueeze(0)  # (1,T)
    action_onehot = F.one_hot(action_labels_full, num_classes=len(actions2idx)).float()  # (1,T,A)

    # ---- node tensors ----
    nodes_norm = [normalize_id(n) for n in nodes]
    try:
        node_ids = [node2idx[n] for n in nodes_norm]
    except KeyError:
        return {"ok": False, "reason": "unknown_node_token", "task": task}

    node_labels_full = torch.tensor(node_ids, device=device).unsqueeze(0)  # (1,T)

    # ---- embeddings sequence ----
    e_seq = torch.tensor(
        np.stack([emb_resolver.get(n) for n in nodes_norm], axis=0),
        device=device
    ).unsqueeze(0)  # (1,T,768)

    # ---- query/target ids ----
    qid = normalize_id(row["query_text_id"]) if "query_text_id" in row else nodes_norm[-2]
    tid = normalize_id(row["target_text_id"]) if "target_text_id" in row else nodes_norm[-1]

    q_text_emb = torch.tensor(emb_resolver.get(qid), device=device).unsqueeze(0)  # (1,768)

    # ---- resolve raw texts (for frozen LM prompt/labels) ----
    q_text = str(id2text.get(qid)).strip()
    t_text = str(id2text.get(tid)).strip()

    # ---- task label ----
    task_labels = torch.tensor([task_id], device=device)

    # ---- sequence loss masking rule ----
    if task == ITEM_TASK:
        keep_len = max(1, T - 2)
    else:
        keep_len = T

    action_labels = action_labels_full.clone()
    node_labels   = node_labels_full.clone()

    if keep_len < T:
        action_labels[0, keep_len:] = IGNORE_INDEX
        node_labels[0, keep_len:]   = IGNORE_INDEX

    # ---- user prediction target/mask ----
    user_mask = torch.tensor([1.0 if task == USER_TASK else 0.0], device=device)
    user_labels = torch.tensor([safe_user_label(row["userid"])], device=device)

    # ---- item prediction target/mask ----
    item_mask = torch.tensor([1.0 if task == ITEM_TASK else 0.0], device=device)
    item_target_id = safe_item_label(tid)
    item_labels = torch.tensor([item_target_id], device=device)

    # ---- word decoder supervision/mask ----
    word_mask = torch.tensor([1.0 if task in GEN_TASKS_WORD else 0.0], device=device)

    word_targets = torch.zeros(1, len(word2idx), device=device)
    gold_words = []
    if task in GEN_TASKS_WORD:
        y, gold_words = build_multi_hot_words(tid, device=device)
        word_targets[0] = y

    return {
        "ok": True,
        "task": task,
        "T": T,

        "e_seq": e_seq,
        "action_onehot": action_onehot,
        "action_labels": action_labels,
        "node_labels": node_labels,

        "task_labels": task_labels,
        "q_text_emb": q_text_emb,

        "user_labels": user_labels,
        "user_mask": user_mask,

        "item_labels": item_labels,
        "item_mask": item_mask,

        "word_targets": word_targets,
        "word_mask": word_mask,
        "gold_words": gold_words,

        "qid": qid,
        "tid": tid,
        "q_text": q_text,
        "t_text": t_text,

        "userid_raw": str(row["userid"]).strip(),
    }


In [14]:
# =========================================================
# CELL 12.9 (NEW) — Loss Utilities (Focal Cross-Entropy with reduction)
# Purpose:
#   - Cross-entropy variant that increases weight on incorrect predictions
#   - Supports reduction='none' for masking
# Inputs:
#   - logits: (B,C)
#   - targets: (B,)
# Outputs:
#   - loss: (B,) if reduction='none' else scalar
# =========================================================

class FocalCELoss(nn.Module):
    def __init__(self, gamma=2.0, reduction="mean"):
        super().__init__()
        assert reduction in {"none", "mean", "sum"}
        self.gamma = float(gamma)
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits: (B,C), targets: (B,)
        logp = F.log_softmax(logits, dim=-1)                  # (B,C)
        p = torch.exp(logp)                                   # (B,C)
        pt = p.gather(1, targets.unsqueeze(1)).squeeze(1)      # (B,)
        logpt = logp.gather(1, targets.unsqueeze(1)).squeeze(1)# (B,)
        loss = -((1.0 - pt) ** self.gamma) * logpt            # (B,)

        if self.reduction == "none":
            return loss
        if self.reduction == "sum":
            return loss.sum()
        return loss.mean()


In [15]:
# =========================================================
# CELL 13 (REPLACE) — UniHis SSM Encoder (same architecture, task loss changed)
# Purpose:
#   - SSM history encoder with task prediction head
#   - Shared sequence heads for action/node (masked via IGNORE_INDEX)
# Inputs:
#   - e_seq, action_onehot, action_labels, node_labels, task_labels, q_text_emb
# Outputs:
#   - losses: loss_action, loss_node, loss_task (FOCAL-CE)
#   - states: h_final (task-conditioned), P_task, logits
# =========================================================

class UniHisSSMEncoder(nn.Module):
    def __init__(
        self,
        vocab_sizes,
        d_model=768,
        d_state=64,
        n_tasks=8,
        d_qtext=768,
        known_task=False
    ):
        super().__init__()
        self.d_state = d_state
        self.known_task = known_task

        # sequence CE (unchanged)
        self.criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

        # task CE (changed to focal CE)
        self.task_criterion = FocalCELoss(gamma=2.0, reduction="mean")

        # ========= SSM EIGEN PARAMS =========
        def init_eig():
            Q = nn.Parameter(torch.randn(d_state, d_state) * 0.02)
            Lambda = nn.Parameter(torch.empty(d_state).uniform_(-1.0, -0.1))
            return Q, Lambda

        self.Q_L, self.Lambda_L = init_eig()
        self.Q_S, self.Lambda_S = init_eig()
        self.Q_E, self.Lambda_E = init_eig()

        self.B = nn.Parameter(torch.randn(d_state, d_model))

        self.tau_S = nn.Parameter(torch.tensor(1.0))
        self.tau_E = nn.Parameter(torch.tensor(1.0))

        # ========= ACTION → MOOD =========
        self.action_proj = nn.Linear(vocab_sizes["actions"], d_model)

        self.MLP_mood = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )

        self.W_a = nn.Linear(d_model, d_state)

        # ========= MIXERS =========
        self.Mixer_L = nn.Linear(d_state, d_state)
        self.Mixer_S = nn.Linear(d_state, d_state)
        self.Mixer_E = nn.Linear(d_state, d_state)

        self.W_fuse = nn.Linear(3 * d_state, 3)
        self.Mixer_Fuse = nn.Linear(d_state, d_state)

        # ========= HEADS =========
        self.action_head = nn.Linear(3 * d_state, vocab_sizes["actions"])
        self.node_head   = nn.Linear(3 * d_state, vocab_sizes["nodes"])

        # ========= TASK =========
        self.Embed_Q = nn.Linear(d_qtext, d_state)
        self.W_task  = nn.Linear(2 * d_state, n_tasks)

        self.W_hyp = nn.Linear(2 * d_state, d_state)
        self.W_theta = nn.ModuleList([
            nn.Linear(d_state, 2 * d_state) for _ in range(n_tasks)
        ])

    # ========= SSM MATRIX =========
    def make_A(self, Q, Lambda_diag):
        Q_inv = torch.linalg.inv(Q)
        return Q @ torch.diag(Lambda_diag) @ Q_inv

    # ========= FORWARD =========
    def forward(
        self,
        e_seq,             # (B, T, d_model)
        action_onehot,     # (B, T, |A|)
        action_labels,     # (B, T) with IGNORE_INDEX for masked steps
        node_labels,       # (B, T) with IGNORE_INDEX for masked steps
        task_labels,       # (B,)
        q_text_emb         # (B, d_qtext)
    ):
        B, T, _ = e_seq.shape
        device = e_seq.device

        # ========= INITIAL STATES =========
        xL = torch.zeros(B, self.d_state, device=device)
        xS = torch.zeros(B, self.d_state, device=device)
        xE = torch.zeros(B, self.d_state, device=device)

        act_logits = []
        node_logits = []

        # ========= SSM MATRICES =========
        A_L = self.make_A(self.Q_L, torch.exp(self.Lambda_L))

        eps_S = torch.sigmoid(self.tau_S)
        k = torch.log((1 - eps_S) / eps_S) / torch.abs(self.Lambda_S)
        A_S = self.make_A(self.Q_S, torch.sigmoid(-k * self.Lambda_S))

        eta = (torch.exp(-1.0 / self.tau_E) - 1.0) / self.Lambda_E
        A_E = self.make_A(self.Q_E, 1.0 + eta * self.Lambda_E)

        # ========= SEQUENCE LOOP =========
        for t in range(T):
            e_t = e_seq[:, t]                      # (B, d_model)
            a_t = action_onehot[:, t, :]           # (B, |A|)

            a_emb = self.action_proj(a_t)
            mood_vec = self.MLP_mood(a_emb)
            mood = torch.sigmoid(self.W_a(mood_vec))  # (B,d_state)

            u_t = torch.bmm(
                (self.B.unsqueeze(0) * mood.unsqueeze(2)),
                e_t.unsqueeze(2)
            ).squeeze(2)  # (B,d_state)

            xL = torch.matmul(xL, A_L.T) + u_t
            xS = torch.matmul(xS, A_S.T) + u_t
            xE = torch.matmul(xE, A_E.T) + u_t

            x_cat = torch.cat([xL, xS, xE], dim=-1)  # (B,3d_state)

            act_logits.append(self.action_head(x_cat))
            node_logits.append(self.node_head(x_cat))

        act_logits  = torch.stack(act_logits, dim=1)     # (B, T, |A|)
        node_logits = torch.stack(node_logits, dim=1)    # (B, T, |V|)

        # ========= HISTORY FUSION =========
        yL = self.Mixer_L(xL)
        yS = self.Mixer_S(xS)
        yE = self.Mixer_E(xE)

        w = F.softmax(self.W_fuse(torch.cat([xL, xS, xE], dim=-1)), dim=-1)
        h_hist = self.Mixer_Fuse(
            w[:, 0:1] * yL +
            w[:, 1:2] * yS +
            w[:, 2:3] * yE
        )

        # ========= ACTION + NODE LOSS (unchanged) =========
        loss_action = self.criterion(
            act_logits.view(-1, act_logits.size(-1)),
            action_labels.view(-1)
        )

        loss_node = self.criterion(
            node_logits.view(-1, node_logits.size(-1)),
            node_labels.view(-1)
        )

        # ========= TASK CONDITIONING =========
        q_emb = self.Embed_Q(q_text_emb)
        z_switch = torch.cat([h_hist, q_emb], dim=-1)

        task_logits = self.W_task(z_switch)
        P_task = F.softmax(task_logits, dim=-1)

        # TASK LOSS (changed)
        loss_task = self.task_criterion(task_logits, task_labels)

        h_hyp = F.relu(self.W_hyp(z_switch))

        if self.known_task:
            theta = self.W_theta[task_labels[0]](h_hyp)
            g, b = theta.chunk(2, dim=-1)
            h_final = torch.sigmoid(g) * h_hist + b
        else:
            h_final = 0.0
            for i in range(len(self.W_theta)):
                theta_i = self.W_theta[i](h_hyp)
                g_i, b_i = theta_i.chunk(2, dim=-1)
                h_final = h_final + P_task[:, i:i+1] * (
                    torch.sigmoid(g_i) * h_hist + b_i
                )

        return {
            "loss_action": loss_action,
            "loss_node": loss_node,
            "loss_task": loss_task,

            "act_logits": act_logits,
            "node_logits": node_logits,

            "task_logits": task_logits,
            "P_task": P_task,

            "h_final": h_final
        }


In [16]:
# =========================================================
# CELL 14 — Task Heads:
#   (A) User MLP head (only for next_user_prediction)
#   (B) Item MLP head (only for next_item_prediction)
# Purpose:
#   - Implement your requested small MLP classifiers
# Inputs:
#   - h_final
# Outputs:
#   - user_logits, item_logits
# =========================================================

class MLPClassifier(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_out)
        )

    def forward(self, x):
        return self.net(x)

class TaskHeads(nn.Module):
    def __init__(self, d_state, n_users_small, n_items, dropout=0.1):
        super().__init__()
        # "proper small mlp" for users
        self.user_head = MLPClassifier(d_state, d_state // 2, n_users_small, dropout=dropout)
        # 2-layer MLP for next-item over item vocab
        self.item_head = MLPClassifier(d_state, d_state, n_items, dropout=dropout)

    def forward(self, h_final):
        return {
            "user_logits": self.user_head(h_final),
            "item_logits": self.item_head(h_final),
        }


In [17]:
# =========================================================
# CELL 15 (REPLACE) — RWKV-style Word Decoder (asymmetric BCE: false positives expensive)
# Purpose:
#   - Word multi-label decoder used only for dialogue_generation + personalized_summarization
#   - Penalize predicting wrong words heavily (high false-positive cost)
#   - Penalize missing gold words lightly (low false-negative cost)
# Inputs:
#   - h_final, q_text_emb, P_task, word_targets, word_mask
# Outputs:
#   - word_logits, loss_word (masked)
# =========================================================

class LensGate(nn.Module):
    def __init__(self, d_state, d_qtext):
        super().__init__()
        self.Wq = nn.Linear(d_qtext, d_state)
        self.Wfilm = nn.Linear(d_state, 2 * d_state)
        self.Wg = nn.Linear(2 * d_state, d_state)

    def forward(self, h_final, q_text_emb):
        q0 = self.Wq(q_text_emb)
        gamma, beta = self.Wfilm(h_final).chunk(2, dim=-1)
        q_lens = (1.0 + gamma) * q0 + beta
        g = torch.sigmoid(self.Wg(torch.cat([h_final, q_lens], dim=-1)))
        return g * q_lens + (1.0 - g) * q0


class RWKVTimeMix(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.time_mix = nn.Parameter(torch.rand(d))
        self.key = nn.Linear(d, d, bias=False)
        self.value = nn.Linear(d, d, bias=False)
        self.receptance = nn.Linear(d, d, bias=False)
        self.output = nn.Linear(d, d, bias=False)
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        x = self.ln(x)
        x_shift = torch.zeros_like(x)
        x_shift[:, 1:] = x[:, :-1]
        xm = x * self.time_mix + x_shift * (1 - self.time_mix)

        r = torch.sigmoid(self.receptance(xm))
        k = self.key(xm)
        v = self.value(xm)

        w = torch.softmax(k, dim=1)
        mix = torch.sum(w * v, dim=1, keepdim=True)
        y = self.output(r * mix).expand_as(x)
        return x + y


class RWKVChannelMix(nn.Module):
    def __init__(self, d, expansion=4):
        super().__init__()
        self.time_mix = nn.Parameter(torch.rand(d))
        self.key = nn.Linear(d, expansion * d, bias=False)
        self.value = nn.Linear(expansion * d, d, bias=False)
        self.receptance = nn.Linear(d, d, bias=False)
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        x = self.ln(x)
        x_shift = torch.zeros_like(x)
        x_shift[:, 1:] = x[:, :-1]
        xm = x * self.time_mix + x_shift * (1 - self.time_mix)

        r = torch.sigmoid(self.receptance(xm))
        k = torch.relu(self.key(xm))
        v = self.value(k)
        return x + r * v


class RWKVWordDecoder(nn.Module):
    def __init__(self, d_state, d_qtext, n_tasks, n_words, n_layers=2,
                 neg_weight=10.0, pos_weight=1.0):
        super().__init__()

        # Asymmetry:
        #   - neg_weight: penalty for false positives (predicting wrong)
        #   - pos_weight: penalty for false negatives (missing gold)
        self.neg_weight = float(neg_weight)
        self.pos_weight = float(pos_weight)

        self.lens_gate = LensGate(d_state, d_qtext)

        self.task_embed = nn.Embedding(n_tasks, d_state)
        self.task_proj = nn.Linear(d_state, d_state)

        self.proj_h = nn.Linear(d_state, d_state)
        self.proj_q = nn.Linear(d_state, d_state)

        self.blocks = nn.ModuleList()
        for _ in range(n_layers):
            self.blocks.append(RWKVTimeMix(d_state))
            self.blocks.append(RWKVChannelMix(d_state))

        self.out_ln = nn.LayerNorm(d_state)
        self.word_head = nn.Linear(d_state, n_words)

    def forward(self, h_final, q_text_emb, P_task, word_targets, word_mask):
        # h_final: (B,d_state)
        # q_text_emb: (B,d_qtext)
        # P_task: (B,n_tasks)
        # word_targets: (B,n_words) multi-hot
        # word_mask: (B,) 1 for gen tasks else 0

        B = h_final.size(0)

        q_cond = self.lens_gate(h_final, q_text_emb)

        task_ids = torch.arange(P_task.size(1), device=h_final.device)
        Temb = self.task_embed(task_ids)                 # (n_tasks,d)
        prompt = self.task_proj(P_task @ Temb)           # (B,d)

        token_h = self.proj_h(h_final + prompt)          # (B,d)
        token_q = self.proj_q(q_cond + prompt)           # (B,d)

        x = torch.stack([token_h, token_q], dim=1)       # (B,2,d)

        for blk in self.blocks:
            x = blk(x)

        h_dec = self.out_ln(x[:, -1])                    # (B,d)
        word_logits = self.word_head(h_dec)              # (B,n_words)

        # ===== Asymmetric BCE (false positives expensive) =====
        # probabilities
        p = torch.sigmoid(word_logits)

        # False positive: target=0 but predicted high -> must be VERY expensive
        fp = -(1.0 - word_targets) * torch.log(1.0 - p + 1e-12)

        # False negative: target=1 but predicted low -> cheap
        fn = -word_targets * torch.log(p + 1e-12)

        # Strong precision bias: fp >> fn
        NEG_WEIGHT = 20.0   # punish hallucinated words
        POS_WEIGHT = 1.0    # tolerate missing gold words

        # apply per-word importance weights (same vocab, same shape)
        # word_importance shape: (n_words,)
        weighted_loss = word_importance.unsqueeze(0) * (NEG_WEIGHT * fp + POS_WEIGHT * fn)

        loss_word_per = weighted_loss.mean(dim=-1)


        # mask non-generation tasks
        loss_word = (loss_word_per * word_mask).sum() / (word_mask.sum() + 1e-12)


        return {
            "word_logits": word_logits,
            "loss_word": loss_word
        }


In [18]:
# =========================================================
# CELL 16 (REPLACE) — Full Multi-Task Model Wrapper + Frozen-LM loss for GEN tasks
# =========================================================

class UniHisMultiTaskModel(nn.Module):
    def __init__(
        self,
        encoder: UniHisSSMEncoder,
        d_qtext: int,
        n_tasks: int,
        n_users_small: int,
        n_items: int,
        n_words: int,
        rwkv_layers: int = 2
    ):
        super().__init__()
        self.encoder = encoder
        self.heads = TaskHeads(d_state=encoder.d_state, n_users_small=n_users_small, n_items=n_items)

        self.word_decoder = RWKVWordDecoder(
            d_state=encoder.d_state,
            d_qtext=d_qtext,
            n_tasks=n_tasks,
            n_words=n_words,
            n_layers=rwkv_layers,
            neg_weight=10.0,
            pos_weight=1.0
        )

        self.user_criterion = FocalCELoss(gamma=2.0, reduction="none")
        self.item_criterion = FocalCELoss(gamma=2.0, reduction="none")

        # ---- Frozen LM is attached later ----
        self.lm = None
        self.lm_tokenizer = None

        # adapter params (created when LM is set)
        self.lm_word_soft_emb = None   # (V_words, d_bridge)
        self.lm_hint_proj = None       # (d_bridge -> hint_len*d_lm)

        # settings
        self.lm_hint_len = 0
        self.lm_d_bridge = 0
        self.lm_d_model = 0
        self.lm_word_temp = 1.0
        self.lm_loss_weight = 1.0
        self.lm_max_prompt_tokens = 256
        self.lm_max_target_tokens = 256

    # ---------- parameter routing for optimizer compatibility ----------
    def base_param_list(self):
        # everything except lm_* params
        return [p for n, p in self.named_parameters() if not n.startswith("lm_")]

    def lm_param_list(self):
        # only lm_* params
        return [p for n, p in self.named_parameters() if n.startswith("lm_")]

    # ---------- attach frozen LM + build differentiable bridge ----------
    def set_frozen_lm(
        self,
        lm_model,
        lm_tokenizer,
        hint_len: int = 8,
        d_bridge: int = 128,
        word_temp: float = 1.0,
        loss_weight: float = 1.0,
        max_prompt_tokens: int = 256,
        max_target_tokens: int = 256,
    ):
        self.lm = lm_model
        self.lm_tokenizer = lm_tokenizer

        self.lm_hint_len = int(hint_len)
        self.lm_d_bridge = int(d_bridge)
        self.lm_d_model = int(self.lm.get_input_embeddings().embedding_dim)

        self.lm_word_temp = float(word_temp)
        self.lm_loss_weight = float(loss_weight)

        self.lm_max_prompt_tokens = int(max_prompt_tokens)
        self.lm_max_target_tokens = int(max_target_tokens)

        # Create differentiable mapping: RWKV word logits -> soft hint embeddings (hint_len tokens)
        Vw = self.word_decoder.word_head.out_features  # n_words
        self.lm_word_soft_emb = nn.Parameter(torch.randn(Vw, self.lm_d_bridge) * 0.02)
        self.lm_hint_proj = nn.Linear(self.lm_d_bridge, self.lm_hint_len * self.lm_d_model)

    def _tokenize_trunc(self, text: str, max_len: int, device):
        enc = self.lm_tokenizer(
            text,
            return_tensors="pt",
            add_special_tokens=False,
            truncation=True,
            max_length=max_len
        )
        return enc["input_ids"].to(device)

    def _build_prompt(self, q_text: str):
        # text-only prompt as requested
        return (
            "Context:\n"
            + (q_text or "")
            + "\n\n"
            "Instruction: Write the best possible response.\n"
            "Response:\n"
        )

    def _frozen_lm_loss(self, word_logits, q_text: str, t_text: str, device):
        """
        Differentiable LM loss:
          - prompt is text tokens
          - RWKV word logits produce soft hint embeddings inserted after prompt
          - gold target text is teacher-forced, loss computed on target tokens only
        """
        if self.lm is None or self.lm_tokenizer is None:
            return None

        q_text = (q_text or "").strip()
        t_text = (t_text or "").strip()
        if len(q_text) == 0 or len(t_text) == 0:
            return None

        prompt = self._build_prompt(q_text)

        prompt_ids = self._tokenize_trunc(prompt, self.lm_max_prompt_tokens, device)
        target_ids = self._tokenize_trunc(t_text, self.lm_max_target_tokens, device)

        if target_ids.numel() == 0:
            return None

        # embeddings
        E = self.lm.get_input_embeddings()
        prompt_embeds = E(prompt_ids)   # (1, Lp, d_lm)
        target_embeds = E(target_ids)   # (1, Lt, d_lm)

        # RWKV -> soft word distribution -> bridge embedding
        # word_logits: (B,Vw); here B=1 in your pipeline
        p_words = torch.softmax(word_logits / self.lm_word_temp, dim=-1)  # (1,Vw)
        e_bridge = p_words @ self.lm_word_soft_emb                          # (1,d_bridge)

        hint_flat = self.lm_hint_proj(e_bridge)                             # (1, hint_len*d_lm)
        hint_embeds = hint_flat.view(1, self.lm_hint_len, self.lm_d_model)  # (1,hint_len,d_lm)

        # full inputs: prompt + hint + gold target
        inputs_embeds = torch.cat([prompt_embeds, hint_embeds, target_embeds], dim=1)

        Lp = prompt_embeds.size(1)
        Lh = hint_embeds.size(1)
        Lt = target_embeds.size(1)
        L = Lp + Lh + Lt

        attention_mask = torch.ones((1, L), device=device, dtype=torch.long)

        # labels: ignore prompt + hint, supervise target only
        labels = torch.full((1, L), -100, device=device, dtype=torch.long)
        labels[:, Lp + Lh:] = target_ids

        out = self.lm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False
        )
        return out.loss  # scalar

    def forward(self, batch):
        enc = self.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h_final = enc["h_final"]
        head_out = self.heads(h_final)

        # ---------- user loss (masked) ----------
        user_logits = head_out["user_logits"]
        user_labels = batch["user_labels"]
        user_mask   = batch["user_mask"]
        loss_user_per = self.user_criterion(user_logits, user_labels)
        loss_user = (loss_user_per * user_mask).sum() / (user_mask.sum() + 1e-8)

        # ---------- item loss (masked) ----------
        item_logits = head_out["item_logits"]
        item_labels = batch["item_labels"]
        item_mask   = batch["item_mask"]
        loss_item_per = self.item_criterion(item_logits, item_labels)
        loss_item = (loss_item_per * item_mask).sum() / (item_mask.sum() + 1e-8)

        # ---------- word loss ----------
        dec = self.word_decoder(
            h_final=enc["h_final"],
            q_text_emb=batch["q_text_emb"],
            P_task=enc["P_task"],
            word_targets=batch["word_targets"],
            word_mask=batch["word_mask"]
        )

        # ---------- frozen LM loss (GEN tasks only) ----------
        loss_lm = None
        if self.lm is not None:
            # compute only if this is a gen row
            if float(batch["word_mask"].item()) > 0.0:
                loss_lm = self._frozen_lm_loss(
                    word_logits=dec["word_logits"],
                    q_text=batch.get("q_text", ""),
                    t_text=batch.get("t_text", ""),
                    device=batch["e_seq"].device
                )

        loss_total = (
            enc["loss_action"]
            + enc["loss_node"]
            + enc["loss_task"]
            + loss_user
            + loss_item
            + dec["loss_word"]
        )

        if loss_lm is not None:
            loss_total = loss_total + self.lm_loss_weight * loss_lm

        return {
            "loss_total": loss_total,

            "loss_action": enc["loss_action"],
            "loss_node": enc["loss_node"],
            "loss_task": enc["loss_task"],
            "loss_user": loss_user,
            "loss_item": loss_item,
            "loss_word": dec["loss_word"],
            "loss_lm": (loss_lm if loss_lm is not None else torch.tensor(0.0, device=h_final.device)),

            "act_logits": enc["act_logits"],
            "node_logits": enc["node_logits"],

            "task_logits": enc["task_logits"],
            "P_task": enc["P_task"],

            "user_logits": user_logits,
            "item_logits": item_logits,
            "word_logits": dec["word_logits"],

            "h_final": enc["h_final"]
        }


In [ ]:
# =========================================================
# REPLACEMENT: CELL 17/17.5/17.6 — Build Frozen LM first, then model, then attach once
# =========================================================

from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- Frozen decoder-only LM ----
LM_NAME = "distilgpt2"  # ~82M params
lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
if lm_tokenizer.pad_token is None:
    lm_tokenizer.pad_token = lm_tokenizer.eos_token

lm_model = AutoModelForCausalLM.from_pretrained(LM_NAME)
lm_model.eval()
for p in lm_model.parameters():
    p.requires_grad = False
lm_model = lm_model.to(device)

print("Frozen LM ready:", LM_NAME, "| d_model:", lm_model.get_input_embeddings().embedding_dim)

# ---- Instantiate Multi-task model (NO LM attachment yet) ----
vocab_sizes = {
    "actions": len(actions2idx),
    "nodes": len(node2idx),
}

encoder = UniHisSSMEncoder(
    vocab_sizes=vocab_sizes,
    d_model=D_MODEL,
    d_state=64,
    n_tasks=n_tasks,
    d_qtext=D_MODEL,
    known_task=False
)

model = UniHisMultiTaskModel(
    encoder=encoder,
    d_qtext=D_MODEL,
    n_tasks=n_tasks,
    n_users_small=len(user2idx_small),
    n_items=len(item2idx),
    n_words=len(word2idx),
    rwkv_layers=2
).to(device)

# ---- Attach frozen LM + build differentiable bridge (lm_* params) ----
model.set_frozen_lm(
    lm_model=lm_model,
    lm_tokenizer=lm_tokenizer,
    hint_len=8,
    d_bridge=128,
    word_temp=1.0,
    loss_weight=1.0,
    max_prompt_tokens=256,
    max_target_tokens=256,
)

# IMPORTANT: model is already on device; LM is already on device
print("Model ready (LM attached).")

def count_parameters(m):
    total = 0
    trainable = 0
    frozen = 0
    for _, p in m.named_parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
        else:
            frozen += n
    return total, trainable, frozen

total_params, trainable_params, frozen_params = count_parameters(model)
print("\n================ MODEL PARAMETER SUMMARY ================")
print(f"Total parameters          : {total_params:,}")
print(f"Trainable parameters      : {trainable_params:,}")
print(f"Frozen parameters         : {frozen_params:,}")


In [ ]:
# # =========================================================
# # CELL 17 — Instantiate Model + Optimizers (base + lm-bridge)
# # (DROP-IN REPLACEMENT)
# # =========================================================

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Device:", device)

# vocab_sizes = {
#     "actions": len(actions2idx),
#     "nodes": len(node2idx),
# }

# encoder = UniHisSSMEncoder(
#     vocab_sizes=vocab_sizes,
#     d_model=D_MODEL,
#     d_state=64,
#     n_tasks=n_tasks,
#     d_qtext=D_MODEL,
#     known_task=False
# )

# model = UniHisMultiTaskModel(
#     encoder=encoder,
#     d_qtext=D_MODEL,
#     n_tasks=n_tasks,
#     n_users_small=len(user2idx_small),
#     n_items=len(item2idx),
#     n_words=len(word2idx),
#     rwkv_layers=2
# ).to(device)

# # Attach frozen LM + build bridge params (lm_*)
# model.set_frozen_lm(
#     lm_model=lm_model,
#     lm_tokenizer=lm_tokenizer,
#     hint_len=8,
#     d_bridge=128,
#     word_temp=1.0,
#     loss_weight=1.0,
#     max_prompt_tokens=256,
#     max_target_tokens=256,
# )

# # Two optimizers: base (resumable), lm-bridge (new)
# optim_base = torch.optim.AdamW(model.base_param_list(), lr=3e-4, weight_decay=1e-2)
# optim_lm   = torch.optim.AdamW(model.lm_param_list(),   lr=3e-4, weight_decay=1e-2)

# print("Model ready.")
# print("Base params:", sum(p.numel() for p in model.base_param_list()))
# print("LM-bridge params:", sum(p.numel() for p in model.lm_param_list()))

# def count_parameters(m):
#     total = 0
#     trainable = 0
#     frozen = 0
#     for _, p in m.named_parameters():
#         n = p.numel()
#         total += n
#         if p.requires_grad:
#             trainable += n
#         else:
#             frozen += n
#     return total, trainable, frozen

# total_params, trainable_params, frozen_params = count_parameters(model)
# print("\n================ MODEL PARAMETER SUMMARY ================")
# print(f"Total deployable parameters : {total_params:,}")
# print(f"Trainable parameters       : {trainable_params:,}")
# print(f"Frozen parameters          : {frozen_params:,}")


In [107]:
# # =========================================================
# # CELL 17.5 — Frozen Decoder-Only LM (50M class)
# # =========================================================

# from transformers import AutoTokenizer, AutoModelForCausalLM

# LM_NAME = "distilgpt2"   # ~82M params, smallest stable GPT-2

# lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
# if lm_tokenizer.pad_token is None:
#     lm_tokenizer.pad_token = lm_tokenizer.eos_token

# lm_model = AutoModelForCausalLM.from_pretrained(LM_NAME)
# lm_model.eval()
# for p in lm_model.parameters():
#     p.requires_grad = False


In [108]:
# # =========================================================
# # CELL 17.6 — Attach frozen LM + build differentiable bridge
# # =========================================================

# model.set_frozen_lm(
#     lm_model=lm_model,
#     lm_tokenizer=lm_tokenizer,
#     hint_len=8,
#     d_bridge=128,
#     word_temp=1.0,
#     loss_weight=1.0,
#     max_prompt_tokens=256,
#     max_target_tokens=256
# )

# # IMPORTANT: move model AFTER attaching LM
# model = model.to(device)
# lm_model = lm_model.to(device)


In [ ]:
# =========================================================
# NEW CELL — Freeze policy for Epoch 5 finetuning
#   Freeze: encoder + item head
#   Train : user head + RWKV word decoder + LM bridge (lm_*)
# =========================================================

def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

# Freeze entire encoder (includes action/node/task heads inside encoder)
set_requires_grad(model.encoder, False)

# Freeze item head only
set_requires_grad(model.heads.item_head, False)

# Train user head
set_requires_grad(model.heads.user_head, True)

# Train RWKV word decoder
set_requires_grad(model.word_decoder, True)

# Train LM-bridge params only (NOT the frozen LM itself)
# lm_word_soft_emb (Parameter) and lm_hint_proj (Linear) are trainable by default.
# Ensure they are trainable explicitly:
if model.lm_word_soft_emb is not None:
    model.lm_word_soft_emb.requires_grad = True
set_requires_grad(model.lm_hint_proj, True)

# Safety: confirm frozen LM is still frozen
for p in model.lm.parameters():
    assert p.requires_grad is False

# Report what will train
trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
frozen = [(n, p.numel()) for n, p in model.named_parameters() if not p.requires_grad]

print("Trainable parameter groups (name, #params):")
for n, k in trainable[:50]:
    print("  ", n, k)
if len(trainable) > 50:
    print("  ... total trainable tensors:", len(trainable))

print("\nTotal trainable params:", sum(k for _, k in trainable))
print("Total frozen params   :", sum(k for _, k in frozen))


In [21]:
# =========================================================
# CELL 18 — Utilities: Top-K decoding with probabilities (User/Item) + Word decoding
# Purpose:
#   - Produce top-15 predictions with probs
# Inputs:
#   - logits tensors
# Outputs:
#   - list[(token, prob)]
# =========================================================

@torch.no_grad()
def topk_probs(logits, idx2tok, k=15):
    probs = torch.softmax(logits, dim=-1)  # (1,V)
    topv, topi = torch.topk(probs, k=min(k, probs.size(-1)), dim=-1)
    topv = topv[0].tolist()
    topi = topi[0].tolist()
    return [(idx2tok[i], float(p)) for i, p in zip(topi, topv)]

@torch.no_grad()
def topk_words_from_logits(word_logits, idx2word, k=20, threshold=0.2):
    # multi-label: use sigmoid
    scores = torch.sigmoid(word_logits)[0]  # (V,)
    keep = scores >= threshold
    idxs = torch.nonzero(keep).squeeze(-1)
    if idxs.numel() == 0:
        # fallback to top-k anyway
        topv, topi = torch.topk(scores, k=min(k, scores.numel()))
        return [(idx2word[int(i)], float(v)) for i, v in zip(topi.tolist(), topv.tolist())]
    vals = scores[idxs]
    kk = min(k, vals.numel())
    topv, order = torch.topk(vals, kk)
    picked = idxs[order]
    return [(idx2word[int(i)], float(v)) for i, v in zip(picked.tolist(), topv.tolist())]


In [ ]:
# =========================================================
# REPLACEMENT: CELL 20 — Training Loop starting Epoch 5
#   - Loads epoch_4.pt
#   - Freezes encoder + item head
#   - Trains only user head + word decoder + LM bridge
# =========================================================

from tqdm.auto import tqdm
import os, re, torch
from collections import defaultdict

CKPT_DIR = "/teamspace/studios/this_studio/checkpoints_v2"
os.makedirs(CKPT_DIR, exist_ok=True)

EPOCHS = 7
CLIP_NORM = 0.75
SAVE_EVERY_ROWS = 12000
SAVE_EVERY_EPOCH = True

BASE_CKPT_PATH = os.path.join(CKPT_DIR, "epoch_4.pt")
if not os.path.exists(BASE_CKPT_PATH):
    raise FileNotFoundError(f"Base checkpoint not found: {BASE_CKPT_PATH}")

LOSS_KEYS_ORDER = [
    "loss_total",
    "loss_action",
    "loss_node",
    "loss_task",
    "loss_user",
    "loss_item",
    "loss_word",
    "loss_lm",
]

LOSS_DECIMALS = {
    "loss_total": 4,
    "loss_user": 4,
    "loss_item": 4,
    "loss_word": 4,
    "loss_lm": 4,
    "loss_action": 2,
    "loss_node": 2,
    "loss_task": 2,
}

def running_mean(x, n):
    return x / max(1, n)

def fmt_loss(name, value, n_ok):
    if n_ok <= 0:
        return "na"
    dec = LOSS_DECIMALS.get(name, 2)
    return f"{running_mean(value, n_ok):.{dec}f}"

def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

# ---- load weights from epoch_4 ----
ckpt = torch.load(BASE_CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model"], strict=False)
model.to(device)

# ---- apply freeze policy again (robust) ----
set_requires_grad(model.encoder, False)
set_requires_grad(model.heads.item_head, False)
set_requires_grad(model.heads.user_head, True)
set_requires_grad(model.word_decoder, True)
if model.lm_word_soft_emb is not None:
    model.lm_word_soft_emb.requires_grad = True
set_requires_grad(model.lm_hint_proj, True)

# ---- optimizer only over trainable params ----
trainable_params = [p for p in model.parameters() if p.requires_grad]
if len(trainable_params) == 0:
    raise RuntimeError("No trainable parameters found. Freeze policy likely wrong.")

optim = torch.optim.AdamW(trainable_params, lr=3e-4, weight_decay=1e-2)

# ---- epoch start: FORCE start from 5 ----
start_epoch = 5
start_row_i = 0
global_step = int(ckpt.get("global_step", 0))

print("STARTING FINETUNE")
print("Loaded ckpt:", BASE_CKPT_PATH)
print("Start epoch:", start_epoch, "| Start row:", start_row_i, "| global_step:", global_step)
print("Trainable tensors:", sum(1 for p in model.parameters() if p.requires_grad))
print("Trainable params :", sum(p.numel() for p in model.parameters() if p.requires_grad))

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()

    sums = {k: 0.0 for k in LOSS_KEYS_ORDER}
    steps_ok = 0
    steps_skip = 0

    row_start = start_row_i if epoch == start_epoch else 0
    pbar = tqdm(range(row_start, len(train_df)), desc=f"Epoch {epoch} (row {row_start}→)")

    for row_i in pbar:
        row = train_df.iloc[row_i]
        batch = build_row_tensors(row, device)

        # inject text for LM (safe even if unused)
        if isinstance(batch, dict) and batch.get("ok", False):
            batch["q_text"] = id2text.get(batch.get("qid", ""), "")
            batch["t_text"] = id2text.get(batch.get("tid", ""), "")

        if not batch["ok"]:
            steps_skip += 1
            continue

        out = model(batch)
        loss = out["loss_total"]

        if not torch.isfinite(loss):
            optim.zero_grad(set_to_none=True)
            steps_skip += 1
            continue

        optim.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(trainable_params, CLIP_NORM)
        optim.step()

        steps_ok += 1
        global_step += 1

        for k in LOSS_KEYS_ORDER:
            if k in out:
                sums[k] += float(out[k].detach().item())

        if (row_i + 1) % SAVE_EVERY_ROWS == 0:
            step_path = os.path.join(CKPT_DIR, f"step_{epoch}_{row_i+1}.pt")
            torch.save({
                "epoch": epoch,
                "row_i_next": row_i + 1,
                "global_step": global_step,
                "model": model.state_dict(),
            }, step_path)

        pbar.set_postfix({
            "L":   fmt_loss("loss_total", sums["loss_total"], steps_ok),
            "La":  fmt_loss("loss_action", sums["loss_action"], steps_ok),
            "Ln":  fmt_loss("loss_node", sums["loss_node"], steps_ok),
            "Lt":  fmt_loss("loss_task", sums["loss_task"], steps_ok),
            "Lu":  fmt_loss("loss_user", sums["loss_user"], steps_ok),
            "Li":  fmt_loss("loss_item", sums["loss_item"], steps_ok),
            "Lw":  fmt_loss("loss_word", sums["loss_word"], steps_ok),
            "Llm": fmt_loss("loss_lm", sums["loss_lm"], steps_ok),
            "skip": steps_skip,
            "row": row_i
        })

    if SAVE_EVERY_EPOCH:
        epoch_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pt")
        torch.save({
            "epoch": epoch + 1,
            "row_i_next": 0,
            "global_step": global_step,
            "model": model.state_dict(),
            "epoch_losses": {k: running_mean(v, steps_ok) for k, v in sums.items()},
            "steps_ok": steps_ok,
            "steps_skip": steps_skip,
        }, epoch_path)

    print("\nEpoch", epoch, "SUMMARY")
    for k in LOSS_KEYS_ORDER:
        if steps_ok > 0:
            dec = LOSS_DECIMALS.get(k, 2)
            print(f"{k:12s}: {running_mean(sums[k], steps_ok):.{dec}f}")
        else:
            print(f"{k:12s}: na")
    print("steps_ok:", steps_ok, "steps_skip:", steps_skip)

    start_row_i = 0


In [ ]:
import torch, math, numpy as np
from collections import defaultdict
from tqdm import tqdm

# ---------------------------------------------------------
# Load trained model (CPU safe)
# ---------------------------------------------------------

CKPT_PATH = "/teamspace/studios/this_studio/checkpoints_v2/epoch_7.pt"

ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=False)
model.to(device)
model.eval()

print("Loaded:", CKPT_PATH, "epoch:", ckpt.get("epoch"), "global_step:", ckpt.get("global_step"))


# =========================================================
# Ranking Metrics
# =========================================================

def mrr_at_k(ranked_ids, gold_id, k=15):
    ranked_ids = ranked_ids[:k]
    for i, rid in enumerate(ranked_ids, start=1):
        if rid == gold_id:
            return 1.0 / i
    return 0.0

def ndcg_at_k(ranked_ids, gold_id, k=15):
    ranked_ids = ranked_ids[:k]
    for i, rid in enumerate(ranked_ids, start=1):
        if rid == gold_id:
            return 1.0 / math.log2(i + 1)
    return 0.0


# =========================================================
# ROUGE (token based, same as your training side)
# =========================================================

def rouge1_f1(pred_tokens, ref_tokens):
    if len(pred_tokens)==0 and len(ref_tokens)==0: return 1.0
    if len(pred_tokens)==0 or len(ref_tokens)==0: return 0.0
    from collections import Counter
    pc = Counter(pred_tokens)
    rc = Counter(ref_tokens)
    overlap = sum((pc & rc).values())
    p = overlap / max(1,len(pred_tokens))
    r = overlap / max(1,len(ref_tokens))
    return 0.0 if p+r==0 else 2*p*r/(p+r)

def lcs_length(a,b):
    n,m=len(a),len(b)
    dp=[0]*(m+1)
    for i in range(1,n+1):
        prev=0
        for j in range(1,m+1):
            tmp=dp[j]
            if a[i-1]==b[j-1]:
                dp[j]=prev+1
            else:
                dp[j]=max(dp[j],dp[j-1])
            prev=tmp
    return dp[m]

def rougeL_f1(pred_tokens, ref_tokens):
    if len(pred_tokens)==0 and len(ref_tokens)==0: return 1.0
    if len(pred_tokens)==0 or len(ref_tokens)==0: return 0.0
    lcs=lcs_length(pred_tokens,ref_tokens)
    p=lcs/max(1,len(pred_tokens))
    r=lcs/max(1,len(ref_tokens))
    return 0.0 if p+r==0 else 2*p*r/(p+r)

def tokenize_simple(x):
    return tokenize_words(x or "")


# =========================================================
# LM generation WITH RWKV hint embeddings
# =========================================================

@torch.no_grad()
def lm_generate_with_hints(word_logits, q_text,
                           max_new_tokens=80,
                           temperature=0.8,
                           top_p=0.95):

    q_text=(q_text or "").strip()
    if len(q_text)==0:
        return ""

    prompt="Context:\n"+q_text+"\n\nInstruction: Write the best possible response.\nResponse:\n"

    tok=model.lm_tokenizer(prompt,return_tensors="pt",
                           truncation=True,
                           max_length=model.lm_max_prompt_tokens,
                           add_special_tokens=False)
    prompt_ids=tok["input_ids"].to(device)
    E=model.lm.get_input_embeddings()
    prompt_embeds=E(prompt_ids)

    p_words=torch.softmax(word_logits/model.lm_word_temp,dim=-1)
    e_bridge=p_words@model.lm_word_soft_emb
    hint_flat=model.lm_hint_proj(e_bridge)
    hint_embeds=hint_flat.view(1,model.lm_hint_len,model.lm_d_model)

    inputs_embeds=torch.cat([prompt_embeds,hint_embeds],dim=1)
    attn=torch.ones((1,inputs_embeds.size(1)),device=device)

    gen_ids=model.lm.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attn,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=model.lm_tokenizer.pad_token_id,
        eos_token_id=model.lm_tokenizer.eos_token_id
    )

    out=model.lm_tokenizer.decode(gen_ids[0],skip_special_tokens=True)
    return out.split("Response:",1)[-1].strip() if "Response:" in out else out.strip()


# =========================================================
# Full Evaluation
# =========================================================

TOPK=15
records=[]
metrics=defaultdict(list)

model.eval()

with torch.no_grad():
    for i in tqdm(range(len(test_df[:150])),desc="Inference"):
        row=test_df.iloc[i]
        batch=build_row_tensors(row,device)
        if not batch["ok"]:
            continue

        batch["q_text"]=id2text.get(batch["qid"],"")
        batch["t_text"]=id2text.get(batch["tid"],"")

        out=model(batch)
        task=batch["task"]

        rec={"task":task,"qid":batch["qid"],"tid":batch["tid"],"userid":batch["userid_raw"]}

        # ----- USER -----
        if task==USER_TASK:
            probs=torch.softmax(out["user_logits"],dim=-1)[0]
            topv,topi=torch.topk(probs,k=min(TOPK,probs.numel()))
            gold=int(batch["user_labels"].item())
            rec["gold_user"]=idx2user_small[gold]
            rec["top15_user"]=[(idx2user_small[int(i)],float(v)) for i,v in zip(topi,topv)]
            metrics["user_mrr15"].append(mrr_at_k(topi.tolist(),gold))
            metrics["user_ndcg15"].append(ndcg_at_k(topi.tolist(),gold))

        # ----- ITEM -----
        if task==ITEM_TASK:
            probs=torch.softmax(out["item_logits"],dim=-1)[0]
            topv,topi=torch.topk(probs,k=min(TOPK,probs.numel()))
            gold=int(batch["item_labels"].item())
            rec["gold_item"]=idx2item[gold]
            rec["top15_item"]=[(idx2item[int(i)],float(v)) for i,v in zip(topi,topv)]
            metrics["item_mrr15"].append(mrr_at_k(topi.tolist(),gold))
            metrics["item_ndcg15"].append(ndcg_at_k(topi.tolist(),gold))

        # ----- GENERATION (LM) -----
        if task in GEN_TASKS_WORD:
            lm_text=lm_generate_with_hints(out["word_logits"],batch["q_text"])
            rec["lm_pred_text"]=lm_text
            rec["gold_text"]=batch["t_text"]

            p=tokenize_simple(lm_text)
            r=tokenize_simple(batch["t_text"])
            metrics[f"{task}_rouge1"].append(rouge1_f1(p,r))
            metrics[f"{task}_rougeL"].append(rougeL_f1(p,r))

        records.append(rec)

results_df=pd.DataFrame(records)

print("\n=== USER ===")
print("MRR@15 :",np.mean(metrics["user_mrr15"]))
print("NDCG@15:",np.mean(metrics["user_ndcg15"]))

print("\n=== ITEM ===")
print("MRR@15 :",np.mean(metrics["item_mrr15"]))
print("NDCG@15:",np.mean(metrics["item_ndcg15"]))

for t in GEN_TASKS_WORD:
    print("\n===",t,"===")
    print("ROUGE-1:",np.mean(metrics[f"{t}_rouge1"]))
    print("ROUGE-L:",np.mean(metrics[f"{t}_rougeL"]))

results_df.head()


In [ ]:
# =========================================================
# CELL 30 — Sanity Check (GEN tasks must have query+target text)
# Purpose:
#   - Verify query_text_id and target_text_id exist and resolve to text
#   - Build headline lookup for docs (so summarization can be "headline gen")
# Inputs:
#   - train_df, id2text, docs_df, main_df columns
# Outputs:
#   - docs_headline (d-* -> Headline)
#   - sanity report printed
# =========================================================

import pandas as pd
import numpy as np
import re

# ---- headline lookup (docs_df) ----
docs_headline = {}
if "NewsID" in docs_df.columns:
    for _, r in docs_df.dropna(subset=["NewsID"]).iterrows():
        cid = to_canonical("d", r["NewsID"])
        hl = str(r.get("Headline", "")).strip()
        if hl:
            docs_headline[cid] = hl

def resolve_text_for_prompt(cid: str) -> str:
    cid = normalize_id(cid)
    # prompt can use the best available text
    # for docs: prefer NewsBody (id2text already stores body if present)
    return str(id2text.get(cid, "")).strip()

def resolve_text_for_gold(task: str, tid: str) -> str:
    """
    Gold supervision text:
      - dialogue_generation: use target text as-is from id2text
      - personalized_summarization: gold should be HEADLINE (as you requested)
        - If tid is a doc (d-*), use docs_headline if available.
        - Else, fall back to id2text(tid).
    """
    task = str(task).strip()
    tid = normalize_id(tid)

    if task == "personalized_summarization":
        if tid.lower().startswith("d-") and tid in docs_headline:
            return docs_headline[tid]
        # fallback
        return str(id2text.get(tid, "")).strip()

    return str(id2text.get(tid, "")).strip()

GEN_TASKS = set(GEN_TASKS_WORD)  # {"dialogue_generation","personalized_summarization"}

need = train_df[train_df["task"].astype(str).str.strip().isin(GEN_TASKS)].copy()

# required columns
assert "query_text_id" in need.columns, "Missing query_text_id column."
assert "target_text_id" in need.columns, "Missing target_text_id column."

# check resolution
need["qid_norm"] = need["query_text_id"].astype(str).map(normalize_id)
need["tid_norm"] = need["target_text_id"].astype(str).map(normalize_id)

need["q_text"] = need["qid_norm"].map(resolve_text_for_prompt)
need["t_gold"] = need.apply(lambda r: resolve_text_for_gold(r["task"], r["tid_norm"]), axis=1)

n_total = len(need)
n_q_ok = int((need["q_text"].str.len() > 0).sum())
n_t_ok = int((need["t_gold"].str.len() > 0).sum())

print("===== SANITY CHECK: GEN TASKS =====")
print("Gen rows:", n_total)
print("Has query text :", f"{n_q_ok}/{n_total}")
print("Has gold text  :", f"{n_t_ok}/{n_total}")

bad_q = need[need["q_text"].str.len() == 0].head(10)[["task","qid_norm","tid_norm"]]
bad_t = need[need["t_gold"].str.len() == 0].head(10)[["task","qid_norm","tid_norm"]]

if len(bad_q):
    print("\n[WARN] Example rows missing query text (first 10):")
    print(bad_q.to_string(index=False))
if len(bad_t):
    print("\n[WARN] Example rows missing gold text (first 10):")
    print(bad_t.to_string(index=False))

# If these warnings are large, fix the dataset mapping first.


In [ ]:
# =========================================================
# CELL 31 — Latest Checkpoint Loader (re-usable for new finetuning)
# Purpose:
#   - Always resume from latest checkpoint in a directory
# Inputs:
#   - CKPT_DIR paths
# Outputs:
#   - helper: load_latest_ckpt(dir) -> (path, ckpt_dict)
# =========================================================

import os
import re
import torch

def load_latest_ckpt(ckpt_dir: str, pattern=r"epoch.*\.pt$"):
    assert os.path.isdir(ckpt_dir), f"Not a directory: {ckpt_dir}"
    files = [f for f in os.listdir(ckpt_dir) if re.search(pattern, f)]
    assert len(files) > 0, f"No checkpoints found in {ckpt_dir} matching {pattern}"

    # prefer larger epoch number if present; else fallback to mtime
    def epoch_key(fn):
        m = re.search(r"epoch[_a-zA-Z]*([0-9]+)\.pt", fn)
        return int(m.group(1)) if m else -1

    files_sorted = sorted(files, key=lambda f: (epoch_key(f), os.path.getmtime(os.path.join(ckpt_dir, f))))
    latest = files_sorted[-1]
    path = os.path.join(ckpt_dir, latest)
    ckpt = torch.load(path, map_location="cpu")
    return path, ckpt

print("Latest-loader ready.")


In [57]:
# =========================================================
# CELL 32 — Replace User Scoring with Bilinear Ranker (+ regularizer)
# Purpose:
#   - Fix user scoring: bilinear scorer with sampled negatives
#   - Provide a ranking loss (InfoNCE over [pos + negatives])
# Inputs:
#   - d_state, n_users_small
# Outputs:
#   - UserBilinearRanker module
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class UserBilinearRanker(nn.Module):
    """
    Scores user u for state h via:
        score(u|h) = (h W) · U[u]
    where:
        h: (B,d)
        W: (d,d)
        U: (n_users,d)
    """
    def __init__(self, d_state: int, n_users: int, init_scale=0.02):
        super().__init__()
        self.d = int(d_state)
        self.n_users = int(n_users)
        self.W = nn.Parameter(torch.randn(self.d, self.d) * init_scale)
        self.U = nn.Embedding(self.n_users, self.d)
        nn.init.normal_(self.U.weight, mean=0.0, std=init_scale)

    def scores_all(self, h):
        # h: (B,d)
        hw = h @ self.W               # (B,d)
        return hw @ self.U.weight.T   # (B,n_users)

    def scores_subset(self, h, user_ids):
        # user_ids: (B,K)
        hw = h @ self.W                           # (B,d)
        Usub = self.U(user_ids)                   # (B,K,d)
        return torch.einsum("bd,bkd->bk", hw, Usub)

    def reg_loss(self):
        # mild L2 regularizer on W and user embeddings
        return 1e-4 * (self.W.pow(2).mean() + self.U.weight.pow(2).mean())

def sampled_user_ranking_loss(ranker: UserBilinearRanker, h, gold_user: torch.LongTensor,
                             n_neg=64):
    """
    h: (B,d)
    gold_user: (B,)
    Uses sampled negatives + softmax CE on subset.
    """
    B = h.size(0)
    device = h.device

    # sample negatives uniformly, avoid matching gold (best-effort)
    neg = torch.randint(low=0, high=ranker.n_users, size=(B, n_neg), device=device)
    gold = gold_user.view(B, 1)
    neg = torch.where(neg == gold, (neg + 1) % ranker.n_users, neg)

    cand = torch.cat([gold, neg], dim=1)          # (B,1+n_neg)
    logits = ranker.scores_subset(h, cand)        # (B,1+n_neg)

    # label is index 0 (the gold in the subset)
    y = torch.zeros(B, dtype=torch.long, device=device)
    loss = F.cross_entropy(logits, y)
    return loss, logits, cand


In [58]:
# =========================================================
# CELL 33 — Inject User State at All Decoder Layers
# Purpose:
#   - Extend RWKVWordDecoder to condition each block on user embedding
#   - For non-user tasks, user conditioning still uses the row's userid (available)
# Inputs:
#   - existing RWKVWordDecoder code structure
# Outputs:
#   - RWKVWordDecoderUserInjected
# =========================================================

class RWKVWordDecoderUserInjected(RWKVWordDecoder):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        d = self.word_head.in_features  # d_state
        self.user_proj = nn.Linear(d, d, bias=False)
        self.user_gate = nn.Linear(2 * d, d)

    def forward(self, h_final, q_text_emb, P_task, word_targets, word_mask, user_vec=None):
        """
        user_vec: (B,d_state) user embedding projected into decoder blocks.
        If None, no user injection is applied.
        """
        B = h_final.size(0)

        q_cond = self.lens_gate(h_final, q_text_emb)

        task_ids = torch.arange(P_task.size(1), device=h_final.device)
        Temb = self.task_embed(task_ids)                 # (n_tasks,d)
        prompt = self.task_proj(P_task @ Temb)           # (B,d)

        token_h = self.proj_h(h_final + prompt)          # (B,d)
        token_q = self.proj_q(q_cond + prompt)           # (B,d)

        x = torch.stack([token_h, token_q], dim=1)       # (B,2,d)

        # user injection vector
        if user_vec is not None:
            u = self.user_proj(user_vec)                 # (B,d)
            # gated residual strength
            g = torch.sigmoid(self.user_gate(torch.cat([h_final, user_vec], dim=-1)))  # (B,d)
            u = g * u
        else:
            u = None

        for blk in self.blocks:
            x = blk(x)
            if u is not None:
                x = x + u.unsqueeze(1)                   # inject at every block

        h_dec = self.out_ln(x[:, -1])                    # (B,d)
        word_logits = self.word_head(h_dec)              # (B,n_words)

        # ----- asymmetric BCE (same as base class) -----
        p = torch.sigmoid(word_logits)
        fp = -(1.0 - word_targets) * torch.log(1.0 - p + 1e-12)
        fn = -word_targets * torch.log(p + 1e-12)

        NEG_WEIGHT = 20.0
        POS_WEIGHT = 1.0

        weighted_loss = word_importance.unsqueeze(0) * (NEG_WEIGHT * fp + POS_WEIGHT * fn)
        loss_word_per = weighted_loss.mean(dim=-1)
        loss_word = (loss_word_per * word_mask).sum() / (word_mask.sum() + 1e-12)

        return {"word_logits": word_logits, "loss_word": loss_word}


In [ ]:
# =========================================================
# CELL 34 — Patch the Model: add ranker + injected decoder + task-specific prompts
# Purpose:
#   - Replace model.word_decoder with injected version
#   - Add model.user_ranker
#   - Make prompt task-specific:
#       dialogue_generation -> "generate a dialogue response"
#       personalized_summarization -> "generate a news headline"
#   - Ensure gold text uses target_text_id mapping (headline for summarization)
# Outputs:
#   - model patched in-place
# =========================================================

import types

# ---- 1) replace word_decoder with injected variant (copy weights where possible) ----
old_dec = model.word_decoder
new_dec = RWKVWordDecoderUserInjected(
    d_state=old_dec.word_head.in_features,
    d_qtext=model.lm_d_bridge if False else D_MODEL,   # not used here; kept consistent with your init
    n_tasks=n_tasks,
    n_words=old_dec.word_head.out_features,
    n_layers=len(old_dec.blocks)//2,
    neg_weight=10.0,
    pos_weight=1.0
).to(device)

# weight transfer (best-effort)
missing, unexpected = new_dec.load_state_dict(old_dec.state_dict(), strict=False)
print("Decoder weight transfer: missing=", len(missing), "unexpected=", len(unexpected))

model.word_decoder = new_dec

# ---- 2) add bilinear user ranker ----
model.user_ranker = UserBilinearRanker(d_state=model.encoder.d_state, n_users=len(user2idx_small)).to(device)

# ---- 3) task-specific prompt builder ----
def _build_prompt_task_specific(self, task: str, q_text: str):
    task = str(task).strip()
    q_text = (q_text or "").strip()

    if task == "dialogue_generation":
        return (
            "Task: Dialogue Response Generation\n"
            "Context:\n"
            + q_text
            + "\n\nInstruction: Write the next assistant response that best continues the dialogue.\n"
            "Response:\n"
        )

    if task == "personalized_summarization":
        return (
            "Task: News Headline Generation\n"
            "Article:\n"
            + q_text
            + "\n\nInstruction: Write a concise, informative news headline for the article.\n"
            "Headline:\n"
        )

    # fallback
    return (
        "Context:\n"
        + q_text
        + "\n\nInstruction: Write the best possible response.\n"
        "Response:\n"
    )

# bind method
model._build_prompt_task_specific = types.MethodType(_build_prompt_task_specific, model)

print("Model patched: injected decoder + bilinear user ranker + task prompts.")


In [ ]:
# =========================================================
# CELL 35 — New Forward Pass for Finetuning (Freeze all except scorer + LM head)
# Purpose:
#   - Implement a finetune-forward that:
#       (A) Freezes everything except:
#           - user_ranker
#           - lm_* (lm_word_soft_emb, lm_hint_proj)
#           - word_decoder user injection params (user_proj, user_gate)
#       (B) Uses:
#           - user ranking loss (upweighted)
#           - LM teacher-forced loss (task-specific prompt)
#           - (optional) small word loss stabilizer
# Notes:
#   - item prediction is frozen and only evaluated later (as requested)
# Outputs:
#   - freeze policy applied + finetune loss function helpers
# =========================================================

def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

# ---- freeze EVERYTHING ----
set_requires_grad(model, False)

# ---- unfreeze scorer + LM bridge + user-injection params ----
set_requires_grad(model.user_ranker, True)

if model.lm_word_soft_emb is not None:
    model.lm_word_soft_emb.requires_grad = True
set_requires_grad(model.lm_hint_proj, True)

# only user injection params inside decoder (keep rest frozen)
set_requires_grad(model.word_decoder.user_proj, True)
set_requires_grad(model.word_decoder.user_gate, True)

# (optional) allow RWKV word head to adapt a bit (often helps)
# If you want STRICT scorer+LM only, set this to False.
ALLOW_WORD_HEAD = True
set_requires_grad(model.word_decoder.word_head, ALLOW_WORD_HEAD)

# sanity: LM itself must remain frozen
for p in model.lm.parameters():
    assert p.requires_grad is False

trainable = [(n, p.numel()) for n,p in model.named_parameters() if p.requires_grad]
print("Trainable tensors:", len(trainable))
print("Trainable params  :", sum(k for _,k in trainable))
print("Sample trainables:")
for n,_ in trainable[:30]:
    print("  ", n)

# ---- helper: task-specific LM loss (uses target gold text and qid text prompt) ----
def frozen_lm_loss_task_specific(model, task: str, word_logits, q_text: str, gold_text: str, device):
    if model.lm is None or model.lm_tokenizer is None:
        return None

    q_text = (q_text or "").strip()
    gold_text = (gold_text or "").strip()
    if len(q_text) == 0 or len(gold_text) == 0:
        return None

    prompt = model._build_prompt_task_specific(task, q_text)

    prompt_ids = model._tokenize_trunc(prompt, model.lm_max_prompt_tokens, device)
    target_ids = model._tokenize_trunc(gold_text, model.lm_max_target_tokens, device)
    if target_ids.numel() == 0:
        return None

    E = model.lm.get_input_embeddings()
    prompt_embeds = E(prompt_ids)
    target_embeds = E(target_ids)

    p_words = torch.softmax(word_logits / model.lm_word_temp, dim=-1)
    e_bridge = p_words @ model.lm_word_soft_emb
    hint_flat = model.lm_hint_proj(e_bridge)
    hint_embeds = hint_flat.view(1, model.lm_hint_len, model.lm_d_model)

    inputs_embeds = torch.cat([prompt_embeds, hint_embeds, target_embeds], dim=1)

    Lp = prompt_embeds.size(1)
    Lh = hint_embeds.size(1)
    Lt = target_embeds.size(1)
    L = Lp + Lh + Lt

    attention_mask = torch.ones((1, L), device=device, dtype=torch.long)
    labels = torch.full((1, L), -100, device=device, dtype=torch.long)
    labels[:, Lp + Lh:] = target_ids

    out = model.lm(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        labels=labels,
        use_cache=False
    )
    return out.loss


In [ ]:
# =========================================================
# CELL 36 — Finetune Loop (resume from latest checkpoint; upweight user-ranking loss)
# Purpose:
#   - Use latest checkpoint (re-usable)
#   - Optimize:
#       loss = LM_W * loss_lm + USER_W * loss_user_rank + WORD_W * loss_word(stabilizer)
#   - Uses STRICT user vocab: user2idx_small from next_user_prediction task only
#   - Uses qid text as prompt; uses tid->gold mapping (headline for summarization)
# Outputs:
#   - finetuned checkpoints saved for reuse
# =========================================================

import os
from collections import defaultdict
from tqdm.auto import tqdm
import torch
from torch.optim import AdamW

# ---- pick resume source ----
# You can point this to your newest finetune directory later.
RESUME_DIR = "/teamspace/studios/this_studio/checkpoints_ft64"  # or checkpoints_v2
save_dir = "/teamspace/studios/this_studio/checkpoints_ft_userfix"
os.makedirs(save_dir, exist_ok=True)

resume_path, resume_ckpt = load_latest_ckpt(RESUME_DIR, pattern=r"\.pt$")
print("Resuming from:", resume_path)

# ---- load weights (strict=False because we added user_ranker + injected decoder params) ----
model.load_state_dict(resume_ckpt["model"], strict=False)
model.to(device)

# re-apply freeze policy (important after load)
set_requires_grad(model, False)
set_requires_grad(model.user_ranker, True)
if model.lm_word_soft_emb is not None:
    model.lm_word_soft_emb.requires_grad = True
set_requires_grad(model.lm_hint_proj, True)
set_requires_grad(model.word_decoder.user_proj, True)
set_requires_grad(model.word_decoder.user_gate, True)
set_requires_grad(model.word_decoder.word_head, ALLOW_WORD_HEAD)

# ---- finetune rows: only GEN tasks + USER task (as you requested) ----
FT_TASKS = set(GEN_TASKS_WORD) | {USER_TASK}
ft_df = train_df[train_df["task"].astype(str).str.strip().isin(FT_TASKS)].reset_index(drop=True)

print("FT rows:", len(ft_df))
print(ft_df["task"].value_counts())

# ---- optimizer over trainable params only ----
trainable_params = [p for p in model.parameters() if p.requires_grad]
optim = AdamW(trainable_params, lr=1e-4, weight_decay=1e-2)

# ---- weights (NOTE: I cannot guarantee "surely" improved metrics; these are reasonable defaults) ----
LM_W   = 8.0
USER_W = 6.0     # upweight user-ranking strongly
WORD_W = 0.05    # small stabilizer

# negatives for ranking
USER_NEG = 128

EPOCHS = 2
CLIP = 0.75
SAVE_EVERY = 12000

global_step = int(resume_ckpt.get("global_step_ft", resume_ckpt.get("global_step", 0)))

model.train()

for ep in range(1, EPOCHS + 1):
    sums = defaultdict(float)
    ok = 0
    skip = 0

    pbar = tqdm(range(len(ft_df)), desc=f"FT_USERFIX Epoch {ep}")

    for i in pbar:
        row = ft_df.iloc[i]
        batch = build_row_tensors(row, device)
        if not batch.get("ok", False):
            skip += 1
            continue

        task = str(batch["task"]).strip()

        # texts:
        q_text = resolve_text_for_prompt(batch["qid"])  # query_text_id mapped text
        gold_text = resolve_text_for_gold(task, batch["tid"])  # target_text_id mapped gold

        # forward encoder (frozen; but we still need h_final)
        with torch.no_grad():
            enc = model.encoder(
                e_seq=batch["e_seq"],
                action_onehot=batch["action_onehot"],
                action_labels=batch["action_labels"],
                node_labels=batch["node_labels"],
                task_labels=batch["task_labels"],
                q_text_emb=batch["q_text_emb"]
            )
        h = enc["h_final"]                 # (1,d_state)
        P_task = enc["P_task"]             # (1,n_tasks)

        # ---- user embedding for injection (always available via userid column) ----
        u_id = safe_user_label(batch["userid_raw"])
        u_id_t = torch.tensor([u_id], device=device, dtype=torch.long)
        user_vec = model.user_ranker.U(u_id_t)  # (1,d_state)

        # ---- losses ----
        loss_user_rank = torch.tensor(0.0, device=device)
        loss_lm = torch.tensor(0.0, device=device)
        loss_word = torch.tensor(0.0, device=device)

        # USER task: ranking loss with negatives
        if task == USER_TASK:
            gold_u = batch["user_labels"]  # strict: user2idx_small
            loss_user_rank, _, _ = sampled_user_ranking_loss(model.user_ranker, h, gold_u, n_neg=USER_NEG)
            loss_user_rank = loss_user_rank + model.user_ranker.reg_loss()

        # GEN tasks: decoder + LM loss (task-specific prompt)
        if task in GEN_TASKS_WORD:
            dec = model.word_decoder(
                h_final=h,
                q_text_emb=batch["q_text_emb"],
                P_task=P_task,
                word_targets=batch["word_targets"],
                word_mask=batch["word_mask"],
                user_vec=user_vec
            )
            loss_word = dec["loss_word"]

            lm_loss_val = frozen_lm_loss_task_specific(
                model=model,
                task=task,
                word_logits=dec["word_logits"],
                q_text=q_text,
                gold_text=gold_text,
                device=device
            )
            if lm_loss_val is not None:
                loss_lm = lm_loss_val

        # total
        loss = LM_W * loss_lm + USER_W * loss_user_rank + WORD_W * loss_word

        if not torch.isfinite(loss):
            optim.zero_grad(set_to_none=True)
            skip += 1
            continue

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, CLIP)
        optim.step()

        ok += 1
        global_step += 1

        sums["loss"] += float(loss.item())
        sums["loss_lm"] += float(loss_lm.item())
        sums["loss_user_rank"] += float(loss_user_rank.item())
        sums["loss_word"] += float(loss_word.item())

        if (i + 1) % SAVE_EVERY == 0:
            path = os.path.join(save_dir, f"step_userfix_e{ep}_{i+1}.pt")
            torch.save({
                "epoch": ep,
                "row_i_next": i + 1,
                "global_step_ft": global_step,
                "model": model.state_dict(),
                "LM_W": LM_W, "USER_W": USER_W, "WORD_W": WORD_W,
                "USER_NEG": USER_NEG,
            }, path)

        pbar.set_postfix({
            "L":   sums["loss"] / max(1, ok),
            "Llm": sums["loss_lm"] / max(1, ok),
            "LuR": sums["loss_user_rank"] / max(1, ok),
            "Lw":  sums["loss_word"] / max(1, ok),
            "skip": skip
        })

    ep_path = os.path.join(save_dir, f"epoch_userfix_{ep}.pt")
    torch.save({
        "epoch": ep + 1,
        "row_i_next": 0,
        "global_step_ft": global_step,
        "model": model.state_dict(),
        "epoch_means": {k: v / max(1, ok) for k, v in sums.items()},
        "steps_ok": ok,
        "steps_skip": skip,
        "LM_W": LM_W, "USER_W": USER_W, "WORD_W": WORD_W,
        "USER_NEG": USER_NEG,
    }, ep_path)

    print("\nEpoch", ep, "means:", {k: round(v / max(1, ok), 6) for k, v in sums.items()}, "ok:", ok, "skip:", skip)

model.eval()
print("Saved to:", save_dir)


In [ ]:
# =========================================================
# CELL 37 — Re-evaluate: user predictions, item predictions, ROUGE (task prompts honored)
# Purpose:
#   - User: use user_ranker scores over all users
#   - Item: unchanged (still uses model.heads.item_head; encoder frozen)
#   - GEN: generate with task-specific prompt + RWKV hints + user injection
# Outputs:
#   - printed metrics + results_df
# =========================================================

import math
from collections import defaultdict
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch

TOPK = 15
# =========================================================
# LOAD LATEST FINETUNE CHECKPOINT (REQUIRED FOR VALID EVAL)
# =========================================================

EVAL_DIR = "/teamspace/studios/this_studio/checkpoints_ft_userfix"   # change if needed

latest_path, latest_ckpt = load_latest_ckpt(EVAL_DIR, pattern=r"\.pt$")
print("\n[USING CHECKPOINT]")
print(latest_path)

# Load weights
model.load_state_dict(latest_ckpt["model"], strict=False)
model = model.to(device)
model.eval()

print("Checkpoint epoch       :", latest_ckpt.get("epoch"))
print("Checkpoint global_step :", latest_ckpt.get("global_step_ft", latest_ckpt.get("global_step")))
print("Loss weights           :", {
    "LM_W": latest_ckpt.get("LM_W"),
    "USER_W": latest_ckpt.get("USER_W"),
    "WORD_W": latest_ckpt.get("WORD_W"),
    "USER_NEG": latest_ckpt.get("USER_NEG"),
})

def mean_or_nan(x):
    return float(np.mean(x)) if len(x) else float("nan")

def mrr_at_k(ranked_ids, gold_id, k=15):
    ranked_ids = ranked_ids[:k]
    for i, rid in enumerate(ranked_ids, start=1):
        if rid == gold_id:
            return 1.0 / i
    return 0.0

def ndcg_at_k(ranked_ids, gold_id, k=15):
    ranked_ids = ranked_ids[:k]
    for i, rid in enumerate(ranked_ids, start=1):
        if rid == gold_id:
            return 1.0 / math.log2(i + 1)
    return 0.0

@torch.no_grad()
def lm_generate_with_hints_task_specific(model, task: str, word_logits, q_text: str,
                                        max_new_tokens=60, temperature=0.8, top_p=0.95):
    q_text = (q_text or "").strip()
    if len(q_text) == 0:
        return ""

    prompt = model._build_prompt_task_specific(task, q_text)

    tok = model.lm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=model.lm_max_prompt_tokens,
        add_special_tokens=False
    )
    prompt_ids = tok["input_ids"].to(device)
    E = model.lm.get_input_embeddings()
    prompt_embeds = E(prompt_ids)

    p_words = torch.softmax(word_logits / model.lm_word_temp, dim=-1)
    e_bridge = p_words @ model.lm_word_soft_emb
    hint_flat = model.lm_hint_proj(e_bridge)
    hint_embeds = hint_flat.view(1, model.lm_hint_len, model.lm_d_model)

    inputs_embeds = torch.cat([prompt_embeds, hint_embeds], dim=1)
    attn = torch.ones((1, inputs_embeds.size(1)), device=device)

    gen_ids = model.lm.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attn,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=model.lm_tokenizer.pad_token_id,
        eos_token_id=model.lm_tokenizer.eos_token_id
    )
    out = model.lm_tokenizer.decode(gen_ids[0], skip_special_tokens=True)

    # strip to the right marker
    if task == "personalized_summarization":
        return out.split("Headline:", 1)[-1].strip() if "Headline:" in out else out.strip()
    return out.split("Response:", 1)[-1].strip() if "Response:" in out else out.strip()

records = []
metrics = defaultdict(list)

model.eval()
with torch.no_grad():
    for i in tqdm(range(len(test_df[:300])), desc="Eval"):
        row = test_df.iloc[i]
        batch = build_row_tensors(row, device)
        if not batch.get("ok", False):
            continue

        task = str(batch["task"]).strip()
        q_text = resolve_text_for_prompt(batch["qid"])
        gold_text = resolve_text_for_gold(task, batch["tid"])

        # encoder
        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )
        h = enc["h_final"]
        P_task = enc["P_task"]

        rec = {"task": task, "qid": batch["qid"], "tid": batch["tid"], "userid": batch["userid_raw"]}

        # USER eval via ranker
        if task == USER_TASK:
            scores = model.user_ranker.scores_all(h)[0]  # (n_users,)
            topv, topi = torch.topk(scores, k=min(TOPK, scores.numel()))
            gold = int(batch["user_labels"].item())
            rec["gold_user"] = idx2user_small[gold]
            rec["top15_user"] = [(idx2user_small[int(j)], float(scores[int(j)].item())) for j in topi.tolist()]
            metrics["user_mrr15"].append(mrr_at_k(topi.tolist(), gold))
            metrics["user_ndcg15"].append(ndcg_at_k(topi.tolist(), gold))

        # ITEM eval unchanged (may not improve because frozen)
        if task == ITEM_TASK:
            head_out = model.heads(h)
            probs = torch.softmax(head_out["item_logits"], dim=-1)[0]
            topv, topi = torch.topk(probs, k=min(TOPK, probs.numel()))
            gold = int(batch["item_labels"].item())
            rec["gold_item"] = idx2item[gold]
            rec["top15_item"] = [(idx2item[int(j)], float(v)) for j, v in zip(topi.tolist(), topv.tolist())]
            metrics["item_mrr15"].append(mrr_at_k(topi.tolist(), gold))
            metrics["item_ndcg15"].append(ndcg_at_k(topi.tolist(), gold))

        # GEN eval: decoder + hints + task-specific prompt
        if task in GEN_TASKS_WORD:
            u_id = safe_user_label(batch["userid_raw"])
            u_id_t = torch.tensor([u_id], device=device, dtype=torch.long)
            user_vec = model.user_ranker.U(u_id_t)

            dec = model.word_decoder(
                h_final=h,
                q_text_emb=batch["q_text_emb"],
                P_task=P_task,
                word_targets=batch["word_targets"],
                word_mask=batch["word_mask"],
                user_vec=user_vec
            )
            pred = lm_generate_with_hints_task_specific(model, task, dec["word_logits"], q_text, max_new_tokens=60)
            rec["lm_pred_text"] = pred
            rec["gold_text"] = gold_text

            p = tokenize_simple(pred)
            r = tokenize_simple(gold_text)

            metrics[f"{task}_rouge1"].append(rouge1_f1(p, r))
            metrics[f"{task}_rougeL"].append(rougeL_f1(p, r))

        records.append(rec)

results_df = pd.DataFrame(records)

print("\n===== USER (ranker) =====")
print("MRR@15 :", mean_or_nan(metrics["user_mrr15"]))
print("NDCG@15:", mean_or_nan(metrics["user_ndcg15"]))

print("\n===== ITEM (frozen) =====")
print("MRR@15 :", mean_or_nan(metrics["item_mrr15"]))
print("NDCG@15:", mean_or_nan(metrics["item_ndcg15"]))

for t in GEN_TASKS_WORD:
    print(f"\n===== {t} (task prompt) =====")
    print("ROUGE-1:", mean_or_nan(metrics[f"{t}_rouge1"]))
    print("ROUGE-L:", mean_or_nan(metrics[f"{t}_rougeL"]))

results_df.head()

In [ ]:
# =========================================================
# CELL 37 — Fully Teacher-Forced Text Evaluation
# =========================================================

import math
from collections import defaultdict
from tqdm import tqdm
import numpy as np
import torch

# =========================================================
# Load latest checkpoint
# =========================================================
EVAL_DIR = "/teamspace/studios/this_studio/checkpoints_ft_userfix"

latest_path, latest_ckpt = load_latest_ckpt(EVAL_DIR, pattern=r"\.pt$")
print("\n[USING CHECKPOINT]")
print(latest_path)

model.load_state_dict(latest_ckpt["model"], strict=False)
model = model.to(device)
model.eval()

print("Checkpoint epoch       :", latest_ckpt.get("epoch"))
print("Checkpoint global_step :", latest_ckpt.get("global_step_ft", latest_ckpt.get("global_step")))

# =========================================================
# Tokenization + Metrics
# =========================================================

def tokenize_simple(text):
    try:
        return tokenize_words(text or "")
    except:
        return [t for t in (text or "").lower().split() if t]

def rouge1_f1(p, r):
    from collections import Counter
    if not p or not r: return 0.0
    pc, rc = Counter(p), Counter(r)
    o = sum((pc & rc).values())
    P = o / len(p)
    R = o / len(r)
    return 0 if P + R == 0 else 2 * P * R / (P + R)

def lcs(a,b):
    n,m=len(a),len(b)
    dp=[0]*(m+1)
    for i in range(n):
        prev=0
        for j in range(m):
            tmp=dp[j+1]
            dp[j+1]=prev+1 if a[i]==b[j] else max(dp[j],dp[j+1])
            prev=tmp
    return dp[m]

def rougeL_f1(p,r):
    if not p or not r: return 0.0
    L=lcs(p,r)
    P=L/len(p)
    R=L/len(r)
    return 0 if P+R==0 else 2*P*R/(P+R)

def corpus_bleu4(refs, hyps):
    try:
        from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
        return corpus_bleu([[r] for r in refs], hyps,
                           weights=(0.25,0.25,0.25,0.25),
                           smoothing_function=SmoothingFunction().method1)
    except:
        return float("nan")

def meteor_avg(refs, hyps):
    try:
        from nltk.translate.meteor_score import meteor_score
        return np.mean([meteor_score([r],h) for r,h in zip(refs,hyps)])
    except:
        return float("nan")

# =========================================================
# FULL generation function
# =========================================================
@torch.no_grad()
def generate_text(model, task, word_logits, q_text, gold_text):

    prompt = model._build_prompt_task_specific(task, q_text)

    tok_p = model.lm_tokenizer(prompt, return_tensors="pt",
                              truncation=True,
                              max_length=model.lm_max_prompt_tokens,
                              add_special_tokens=False)

    tok_t = model.lm_tokenizer(gold_text, return_tensors="pt",
                              truncation=True,
                              max_length=model.lm_max_target_tokens,
                              add_special_tokens=False)

    prompt_ids = tok_p["input_ids"].to(device)
    target_ids = tok_t["input_ids"].to(device)

    if target_ids.numel() == 0:
        return "", None

    E = model.lm.get_input_embeddings()
    prompt_emb = E(prompt_ids)
    target_emb = E(target_ids)

    # RWKV hint
    p_words = torch.softmax(word_logits / model.lm_word_temp, dim=-1)
    e_bridge = p_words @ model.lm_word_soft_emb
    hint_flat = model.lm_hint_proj(e_bridge)
    hint_emb = hint_flat.view(1, model.lm_hint_len, model.lm_d_model)

    inputs = torch.cat([prompt_emb, hint_emb, target_emb], dim=1)

    Lp, Lh, Lt = prompt_emb.size(1), hint_emb.size(1), target_emb.size(1)
    labels = torch.full((1, Lp + Lh + Lt), -100, device=device)
    labels[:, Lp + Lh:] = target_ids

    attn = torch.ones_like(labels)

    out = model.lm(inputs_embeds=inputs, attention_mask=attn,
                   labels=labels, use_cache=False)

    logits = out.logits[:, (Lp+Lh-1):(Lp+Lh-1+Lt)]
    pred_ids = torch.argmax(logits, dim=-1)

    text = model.lm_tokenizer.decode(pred_ids[0], skip_special_tokens=True)
    return text, float(out.loss.item())

# =========================================================
# Run evaluation
# =========================================================

metrics = defaultdict(list)
dlg_refs, dlg_hyps = [], []
sum_refs, sum_hyps = [], []

model.eval()
with torch.no_grad():
    for i in tqdm(range(len(test_df[300:1300])), desc="Inference Generations"):
        row = test_df.iloc[i]
        batch = build_row_tensors(row, device)
        if not batch.get("ok", False):
            continue

        task = str(batch["task"])
        q_text = resolve_text_for_prompt(batch["qid"])
        gold_text = resolve_text_for_gold(task, batch["tid"])

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h = enc["h_final"]
        P_task = enc["P_task"]

        u_id = safe_user_label(batch["userid_raw"])
        user_vec = model.user_ranker.U(torch.tensor([u_id], device=device))

        dec = model.word_decoder(
            h_final=h,
            q_text_emb=batch["q_text_emb"],
            P_task=P_task,
            word_targets=batch["word_targets"],
            word_mask=batch["word_mask"],
            user_vec=user_vec
        )

        pred, ce = generate_text(model, task, dec["word_logits"], q_text, gold_text)

        if ce is not None:
            metrics[f"{task}_ce"].append(ce)

        pt = tokenize_simple(pred)
        rt = tokenize_simple(gold_text)

        metrics[f"{task}_rouge1"].append(rouge1_f1(pt,rt))
        metrics[f"{task}_rougeL"].append(rougeL_f1(pt,rt))

        if task=="dialogue_generation":
            dlg_refs.append(rt); dlg_hyps.append(pt)
        if task=="personalized_summarization":
            sum_refs.append(rt); sum_hyps.append(pt)

# =========================================================
# Print summary
# =========================================================
print("\n================ TEACHER-FORCED RESULTS ================")

for t in GEN_TASKS_WORD:
    ce = np.mean(metrics[f"{t}_ce"])
    print(f"\n===== {t} =====")
    print("Cross-Entropy :", ce)
    print("Perplexity    :", math.exp(ce))
    print("ROUGE-1       :", np.mean(metrics[f"{t}_rouge1"]))
    print("ROUGE-L       :", np.mean(metrics[f"{t}_rougeL"]))

    if t=="dialogue_generation":
        print("BLEU-4        :", corpus_bleu4(dlg_refs, dlg_hyps))
        print("METEOR       :", meteor_avg(
            [" ".join(r) for r in dlg_refs],
            [" ".join(h) for h in dlg_hyps]
        ))

    if t=="personalized_summarization":
        print("BLEU-4        :", corpus_bleu4(sum_refs, sum_hyps))

print("\n========================================================")


In [ ]:
# =========================================================
# CELL 38 — Resume Latest Checkpoint + Add "Direct Encoder→Hint Bridge"
# Goal:
#   - Load latest checkpoint (strict=False)
#   - Freeze RWKV word decoder entirely (unused in this phase)
#   - Add a NEW trainable bridge: h_final (d_state) -> hint embeddings (hint_len * d_lm)
#   - Ensure NO shape mismatch (safe, strict=False)
# =========================================================

import os, torch, types
import torch.nn as nn
import torch.nn.functional as F

# ----- resume from latest checkpoint -----
RESUME_DIR = "/teamspace/studios/this_studio/checkpoints_ft_userfix"   # change if you want
latest_path, latest_ckpt = load_latest_ckpt(RESUME_DIR, pattern=r"\.pt$")
print("[RESUME]", latest_path)

model.load_state_dict(latest_ckpt["model"], strict=False)
model = model.to(device)
model.eval()

print("Loaded epoch:", latest_ckpt.get("epoch"), "global_step:", latest_ckpt.get("global_step_ft", latest_ckpt.get("global_step")))

# ----- helper freeze/unfreeze -----
def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

# ----- RWKV is NOT used; freeze it hard -----
if hasattr(model, "word_decoder") and model.word_decoder is not None:
    set_requires_grad(model.word_decoder, False)

# ----- add NEW bridge (trainable): h_final -> hint embeddings -----
# We keep name unique to avoid collisions with old lm_* params.
# Output = hint_len * lm_d_model, then reshaped to (1, hint_len, lm_d_model)

if not hasattr(model, "enc2hint_proj") or model.enc2hint_proj is None:
    model.enc2hint_proj = nn.Sequential(
        nn.LayerNorm(model.encoder.d_state),
        nn.Linear(model.encoder.d_state, model.lm_hint_len * model.lm_d_model)
    ).to(device)
    print("Created model.enc2hint_proj")

# Make sure new module params are trainable by default
set_requires_grad(model.enc2hint_proj, True)

print("Bridge ready:",
      "d_state=", model.encoder.d_state,
      "hint_len=", model.lm_hint_len,
      "lm_d_model=", model.lm_d_model)


In [ ]:
# =========================================================
# CELL 39 — Unfreeze Policy for This Phase
# You requested:
#   - Encoder FINETUNING (including action/node predictions)
#   - RWKV frozen (already done)
#   - User head finetuned (model.heads.user_head)
#   - Bridge trainable (enc2hint_proj)
#   - Trainable: last 3 transformer blocks + LM head of GPT (distilgpt2)
# Notes:
#   - distilgpt2 has 6 blocks -> last 3 are blocks [3], [4], [5]
#   - LM head is `model.lm.lm_head` (tied weights in GPT2-style models)
# =========================================================

# 1) Freeze everything first
set_requires_grad(model, False)

# 2) Unfreeze encoder (full encoder, includes action/node/task heads)
set_requires_grad(model.encoder, True)

# 3) Unfreeze user head only (item head stays frozen)
if hasattr(model, "heads") and model.heads is not None:
    set_requires_grad(model.heads, False)
    set_requires_grad(model.heads.user_head, True)
    # keep item frozen explicitly
    set_requires_grad(model.heads.item_head, False)

# 4) Unfreeze user_ranker if you want it evaluated/trained; you asked "user head",
#    so by default we keep user_ranker frozen (it is a different head).
#    If you ALSO want user_ranker trained, uncomment:
# set_requires_grad(model.user_ranker, True)

# 5) Unfreeze bridge
set_requires_grad(model.enc2hint_proj, True)

# 6) Unfreeze last 3 transformer blocks + LM head
#    Keep embeddings + earlier blocks frozen.
assert hasattr(model, "lm") and model.lm is not None, "Frozen LM not attached."

# freeze all LM first (already frozen by set_requires_grad(model, False), but keep explicit)
set_requires_grad(model.lm, False)

# unfreeze last 3 blocks
if hasattr(model.lm, "transformer") and hasattr(model.lm.transformer, "h"):
    blocks = model.lm.transformer.h
    n_blocks = len(blocks)
    last_k = 3
    start = max(0, n_blocks - last_k)
    for bi in range(start, n_blocks):
        set_requires_grad(blocks[bi], True)
    print(f"Unfroze LM blocks: {start}..{n_blocks-1} (total {n_blocks})")
else:
    raise RuntimeError("Unexpected LM structure: model.lm.transformer.h not found.")

# unfreeze lm_head
if hasattr(model.lm, "lm_head"):
    set_requires_grad(model.lm.lm_head, True)
else:
    raise RuntimeError("LM head not found at model.lm.lm_head")

# optional: final layernorm may help; if present, unfreeze it
if hasattr(model.lm, "transformer") and hasattr(model.lm.transformer, "ln_f"):
    set_requires_grad(model.lm.transformer.ln_f, True)

# 7) confirm RWKV stays frozen
if hasattr(model, "word_decoder") and model.word_decoder is not None:
    for p in model.word_decoder.parameters():
        assert p.requires_grad is False

# Report
trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
frozen = [(n, p.numel()) for n, p in model.named_parameters() if not p.requires_grad]

print("\n================ TRAINABLE SUMMARY ================")
print("Trainable tensors:", len(trainable))
print("Trainable params  :", sum(k for _, k in trainable))
print("Frozen params     :", sum(k for _, k in frozen))
print("\nSample trainables:")
for n, k in trainable[:50]:
    print(" ", n, k)
if len(trainable) > 50:
    print(" ... total trainable tensors:", len(trainable))


In [76]:
# =========================================================
# CELL 40 — New LM Teacher-Forced Loss Using Encoder→Hint Bridge (No RWKV)
# Goal:
#   - Build prompt via model._build_prompt_task_specific(task, q_text)
#   - Create hint embeddings directly from h_final using model.enc2hint_proj
#   - Teacher-force gold tokens and compute CE on target tokens only
# =========================================================

@torch.no_grad()
def _tokenize_trunc(tokenizer, text, max_len, device):
    enc = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=int(max_len)
    )
    return enc["input_ids"].to(device)

def enc_bridge_hint_embeds(model, h_final):
    """
    h_final: (B,d_state) expected B=1 in your row-by-row pipeline
    returns: (B, hint_len, lm_d_model)
    """
    flat = model.enc2hint_proj(h_final)  # (B, hint_len*lm_d_model)
    return flat.view(h_final.size(0), model.lm_hint_len, model.lm_d_model)

def teacher_forced_lm_loss_from_h(model, task, h_final, q_text, gold_text, device):
    q_text = (q_text or "").strip()
    gold_text = (gold_text or "").strip()
    if len(q_text) == 0 or len(gold_text) == 0:
        return None

    prompt = model._build_prompt_task_specific(task, q_text)

    prompt_ids = _tokenize_trunc(model.lm_tokenizer, prompt, model.lm_max_prompt_tokens, device)
    target_ids = _tokenize_trunc(model.lm_tokenizer, gold_text, model.lm_max_target_tokens, device)
    if target_ids.numel() == 0:
        return None

    E = model.lm.get_input_embeddings()
    prompt_emb = E(prompt_ids)        # (1,Lp,d)
    target_emb = E(target_ids)        # (1,Lt,d)
    hint_emb   = enc_bridge_hint_embeds(model, h_final)  # (1,Lh,d)

    inputs = torch.cat([prompt_emb, hint_emb, target_emb], dim=1)

    Lp, Lh, Lt = prompt_emb.size(1), hint_emb.size(1), target_emb.size(1)
    labels = torch.full((1, Lp + Lh + Lt), -100, device=device, dtype=torch.long)
    labels[:, Lp + Lh:] = target_ids

    attn = torch.ones_like(labels)

    out = model.lm(inputs_embeds=inputs, attention_mask=attn, labels=labels, use_cache=False)
    return out.loss  # scalar


In [ ]:
# =========================================================
# CELL 41 — Finetune Loop with True Dynamic Resume + Live Losses
# =========================================================

from tqdm.auto import tqdm
from torch.optim import AdamW
from collections import defaultdict
import numpy as np
import math, os, re, torch
import torch.nn.functional as F

SAVE_DIR = "/teamspace/studios/this_studio/checkpoints_encbridge_gpt"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------ Loss Weights ------------------
W_ACTION = 1.0
W_NODE   = 1.0
W_TASK   = 0.2
W_USER   = 1.0
W_LM     = 1.0

EPOCHS = 3
CLIP_NORM = 1.0
LR = 3e-3

# =========================================================
# Resume selector
# =========================================================
def select_resume_state(folder):
    best_epoch = (-1, None)
    best_step  = (-1, -1, None)

    for f in os.listdir(folder):
        if f.endswith(".pt"):
            m = re.match(r"epoch_(\d+)\.pt", f)
            if m:
                best_epoch = max(best_epoch, (int(m.group(1)), f), key=lambda x: x[0])
            m = re.match(r"step_e(\d+)_(\d+)\.pt", f)
            if m:
                best_step = max(best_step, (int(m.group(1)), int(m.group(2)), f),
                                key=lambda x: (x[0], x[1]))

    if best_step[2] is not None and best_step[0] > best_epoch[0]:
        ckpt_file = best_step[2]
    elif best_epoch[1] is not None:
        ckpt_file = best_epoch[1]
    else:
        return None, 1, 0, 0

    ckpt = torch.load(os.path.join(folder, ckpt_file), map_location="cpu")

    start_epoch = int(ckpt.get("epoch", 1))
    start_row   = int(ckpt.get("row_i_next", 0))
    global_step = int(ckpt.get("global_step_encbridge", 0))

    print("[RESUME]", ckpt_file)
    print(" epoch      :", start_epoch)
    print(" row_i_next :", start_row)
    print(" global_step:", global_step)

    model.load_state_dict(ckpt["model"], strict=False)
    return ckpt_file, start_epoch, start_row, global_step


# =========================================================
# Load resume state
# =========================================================
_, start_epoch, start_row, global_step = select_resume_state(SAVE_DIR)

# ------------------ Optimizer ------------------
trainable_params = [p for p in model.parameters() if p.requires_grad]
optim = AdamW(trainable_params, lr=LR, weight_decay=1e-2)
print("Trainable params:", sum(p.numel() for p in trainable_params))

# =========================================================
# Training Loop
# =========================================================
for ep in range(start_epoch, EPOCHS + 1):

    row_begin = start_row if ep == start_epoch else 0
    print(f"\n[Epoch {ep}] starting at row {row_begin}")

    model.train()
    sums = defaultdict(float)
    ok = 0
    skip = 0

    pbar = tqdm(range(row_begin, len(train_df)), desc=f"ENC+GPT FT Epoch {ep}")

    for i in pbar:
        batch = build_row_tensors(train_df.iloc[i], device)
        if not batch.get("ok", False):
            skip += 1
            continue

        task = str(batch["task"]).strip()
        q_text = resolve_text_for_prompt(batch["qid"])
        gold_text = resolve_text_for_gold(task, batch["tid"])

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        loss_action = enc["loss_action"]
        loss_node   = enc["loss_node"]
        loss_task   = enc["loss_task"]

        loss_user = torch.tensor(0.0, device=device)
        if task == USER_TASK:
            loss_user = F.cross_entropy(model.heads.user_head(enc["h_final"]),
                                        batch["user_labels"])

        loss_lm = torch.tensor(0.0, device=device)
        if task in GEN_TASKS_WORD:
            lm = teacher_forced_lm_loss_from_h(model, task, enc["h_final"],
                                               q_text, gold_text, device)
            if lm is not None:
                loss_lm = lm

        loss = (W_ACTION*loss_action + W_NODE*loss_node + W_TASK*loss_task +
                W_USER*loss_user + W_LM*loss_lm)

        if not torch.isfinite(loss):
            skip += 1
            continue

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, CLIP_NORM)
        optim.step()

        ok += 1
        global_step += 1

        sums["loss"] += loss.item()
        sums["action"] += loss_action.item()
        sums["node"] += loss_node.item()
        sums["task"] += loss_task.item()
        sums["user"] += loss_user.item()
        sums["lm"] += loss_lm.item()

        # ----- live tqdm display -----
        pbar.set_postfix({
            "L": sums["loss"]/max(1,ok),
            "La": sums["action"]/max(1,ok),
            "Ln": sums["node"]/max(1,ok),
            "Lt": sums["task"]/max(1,ok),
            "Lu": sums["user"]/max(1,ok),
            "Llm": sums["lm"]/max(1,ok),
            "skip": skip
        })

        # ----- save step checkpoint -----
        if (i+1) % 12000 == 0:
            torch.save({
                "epoch": ep,
                "row_i_next": i+1,
                "global_step_encbridge": global_step,
                "model": model.state_dict()
            }, os.path.join(SAVE_DIR, f"step_e{ep}_{i+1}.pt"))

    # ----- end-of-epoch checkpoint -----
    torch.save({
        "epoch": ep+1,
        "row_i_next": 0,
        "global_step_encbridge": global_step,
        "model": model.state_dict()
    }, os.path.join(SAVE_DIR, f"epoch_{ep}.pt"))

    print("\n[Epoch", ep, "means]",
          {k: round(v/max(1,ok),6) for k,v in sums.items()},
          "ok:", ok, "skip:", skip)

model.eval()
print("\nTraining complete.")


In [ ]:
# =========================================================
# CELL 42 — Evaluation (All Tasks, TF + AR ROUGE)
#   Uses the SAME resume logic as training
# =========================================================

import pandas as pd
import numpy as np
import math, os, re, torch
from collections import defaultdict
from tqdm import tqdm

EVAL_DIR = "/teamspace/studios/this_studio/checkpoints_encbridge_gpt"

# =========================================================
# Same selector as CELL-41
# =========================================================
def select_resume_state(folder):
    best_epoch = (-1, None)
    best_step  = (-1, -1, None)

    for f in os.listdir(folder):
        if f.endswith(".pt"):
            m = re.match(r"epoch_(\d+)\.pt", f)
            if m:
                best_epoch = max(best_epoch, (int(m.group(1)), f), key=lambda x: x[0])
            m = re.match(r"step_e(\d+)_(\d+)\.pt", f)
            if m:
                best_step = max(best_step, (int(m.group(1)), int(m.group(2)), f),
                                key=lambda x: (x[0], x[1]))

    if best_step[2] is not None and best_step[0] > best_epoch[0]:
        ckpt_file = best_step[2]
    elif best_epoch[1] is not None:
        ckpt_file = best_epoch[1]
    else:
        raise RuntimeError("No checkpoint found")

    ckpt = torch.load(os.path.join(folder, ckpt_file), map_location="cpu")
    print("[EVAL USING]", ckpt_file)
    return ckpt


# =========================================================
# Load correct checkpoint
# =========================================================
latest_ckpt = select_resume_state(EVAL_DIR)
model.load_state_dict(latest_ckpt["model"], strict=False)
model = model.to(device)
model.eval()

# =========================================================
# Helpers
# =========================================================
def mean_or_nan(x):
    return float(np.mean(x)) if len(x) else float("nan")

def mrr_at_k(ranked_ids, gold_id):
    for i, rid in enumerate(ranked_ids, 1):
        if rid == gold_id:
            return 1.0 / i
    return 0.0

def ndcg_at_k(ranked_ids, gold_id):
    for i, rid in enumerate(ranked_ids, 1):
        if rid == gold_id:
            return 1.0 / math.log2(i + 1)
    return 0.0



# =========================================================
# LM helpers
# =========================================================
@torch.no_grad()
def lm_generate_from_h(model, task, h_final, q_text, max_new_tokens=60):
    q_text = (q_text or "").strip()
    if not q_text:
        return ""

    prompt = model._build_prompt_task_specific(task, q_text)
    tok = model.lm_tokenizer(prompt, return_tensors="pt", truncation=True,
                             max_length=model.lm_max_prompt_tokens,
                             add_special_tokens=False)

    E = model.lm.get_input_embeddings()
    prompt_emb = E(tok["input_ids"].to(device))
    hint_emb = enc_bridge_hint_embeds(model, h_final)

    inputs = torch.cat([prompt_emb, hint_emb], dim=1)
    attn = torch.ones((1, inputs.size(1)), device=device)

    out = model.lm.generate(
        inputs_embeds=inputs,
        attention_mask=attn,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=model.lm_tokenizer.pad_token_id,
        eos_token_id=model.lm_tokenizer.eos_token_id
    )

    txt = model.lm_tokenizer.decode(out[0], skip_special_tokens=True)
    if task == "personalized_summarization":
        return txt.split("Headline:",1)[-1].strip()
    return txt.split("Response:",1)[-1].strip()


@torch.no_grad()
def teacher_forced_decode(model, task, h, q_text, gold_text):
    if not q_text or not gold_text:
        return "", None

    prompt = model._build_prompt_task_specific(task, q_text)
    p_ids = _tokenize_trunc(model.lm_tokenizer, prompt, model.lm_max_prompt_tokens, device)
    t_ids = _tokenize_trunc(model.lm_tokenizer, gold_text, model.lm_max_target_tokens, device)

    if t_ids.numel() == 0:
        return "", None

    E = model.lm.get_input_embeddings()
    prompt_emb = E(p_ids)
    target_emb = E(t_ids)
    hint_emb = enc_bridge_hint_embeds(model, h)

    inp = torch.cat([prompt_emb, hint_emb, target_emb], dim=1)

    Lp, Lh, Lt = prompt_emb.size(1), hint_emb.size(1), target_emb.size(1)
    labels = torch.full((1, Lp+Lh+Lt), -100, device=device)
    labels[:, Lp+Lh:] = t_ids

    out = model.lm(inputs_embeds=inp, labels=labels)

    logits = out.logits[:, Lp+Lh-1:Lp+Lh-1+Lt]
    pred_ids = logits.argmax(-1)
    pred = model.lm_tokenizer.decode(pred_ids[0], skip_special_tokens=True)

    return pred, float(out.loss.item())


# =========================================================
# Evaluation loop
# =========================================================
TOPK = 15
metrics = defaultdict(list)
records = []

with torch.no_grad():
    for i in tqdm(range(len(test_df[:300]))):
        batch = build_row_tensors(test_df.iloc[i], device)
        if not batch.get("ok"): continue

        task = batch["task"]
        q = resolve_text_for_prompt(batch["qid"])
        gold = resolve_text_for_gold(task, batch["tid"])

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h = enc["h_final"]
        rec = {"task": task, "qid": batch["qid"], "gold": gold}

        # ---------- USER ----------
        if task == USER_TASK:
            probs = torch.softmax(model.heads.user_head(h), -1)[0]
            top = probs.topk(TOPK).indices
            gold_id = int(batch["user_labels"])
            metrics["user_mrr"].append(mrr_at_k(top.tolist(), gold_id))
            metrics["user_ndcg"].append(ndcg_at_k(top.tolist(), gold_id))

        # ---------- ITEM ----------
        if task == ITEM_TASK:
            probs = torch.softmax(model.heads.item_head(h), -1)[0]
            top = probs.topk(TOPK).indices
            gold_id = int(batch["item_labels"])
            metrics["item_mrr"].append(mrr_at_k(top.tolist(), gold_id))
            metrics["item_ndcg"].append(ndcg_at_k(top.tolist(), gold_id))

        # ---------- GENERATION ----------
        if task in GEN_TASKS_WORD:
            tf_pred, ce = teacher_forced_decode(model, task, h, q, gold)
            ar_pred = lm_generate_from_h(model, task, h, q)

            ref_tok = tokenize_simple(gold)
            tf_tok = tokenize_simple(tf_pred)
            ar_tok = tokenize_simple(ar_pred)

            metrics[f"{task}_tf_rouge1"].append(rouge1_f1(tf_tok, ref_tok))
            metrics[f"{task}_tf_rougeL"].append(rougeL_f1(tf_tok, ref_tok))
            metrics[f"{task}_ar_rouge1"].append(rouge1_f1(ar_tok, ref_tok))
            metrics[f"{task}_ar_rougeL"].append(rougeL_f1(ar_tok, ref_tok))

            if ce is not None:
                metrics[f"{task}_ce"].append(ce)

            rec["tf_pred"] = tf_pred
            rec["ar_pred"] = ar_pred

        records.append(rec)

results_df = pd.DataFrame(records)

# =========================================================
# Summary
# =========================================================
print("\n===== USER =====")
print("MRR:", mean_or_nan(metrics["user_mrr"]))
print("NDCG:", mean_or_nan(metrics["user_ndcg"]))

print("\n===== ITEM =====")
print("MRR:", mean_or_nan(metrics["item_mrr"]))
print("NDCG:", mean_or_nan(metrics["item_ndcg"]))

for t in GEN_TASKS_WORD:
    print(f"\n===== {t} =====")
    ce = mean_or_nan(metrics[f"{t}_ce"])
    print("TF CE:", ce, "PPL:", math.exp(ce))
    print("TF ROUGE-1:", mean_or_nan(metrics[f"{t}_tf_rouge1"]))
    print("TF ROUGE-L:", mean_or_nan(metrics[f"{t}_tf_rougeL"]))
    print("AR ROUGE-1:", mean_or_nan(metrics[f"{t}_ar_rouge1"]))
    print("AR ROUGE-L:", mean_or_nan(metrics[f"{t}_ar_rougeL"]))

results_df.head()


In [ ]:
# =========================================================
# CELL 43 — One qualitative sample per task
#   Uses the SAME epoch-vs-step resume logic
# =========================================================

import os, re, torch
from collections import defaultdict

EVAL_DIR = "/teamspace/studios/this_studio/checkpoints_encbridge_gpt"

# -------- same selector --------
def select_resume_state(folder):
    best_epoch = (-1, None)
    best_step  = (-1, -1, None)

    for f in os.listdir(folder):
        if f.endswith(".pt"):
            m = re.match(r"epoch_(\d+)\.pt", f)
            if m:
                best_epoch = max(best_epoch, (int(m.group(1)), f), key=lambda x: x[0])
            m = re.match(r"step_e(\d+)_(\d+)\.pt", f)
            if m:
                best_step = max(best_step, (int(m.group(1)), int(m.group(2)), f),
                                key=lambda x: (x[0], x[1]))

    if best_step[2] is not None and best_step[0] > best_epoch[0]:
        ckpt_file = best_step[2]
    else:
        ckpt_file = best_epoch[1]

    ckpt = torch.load(os.path.join(folder, ckpt_file), map_location="cpu")
    print("[USING]", ckpt_file)
    return ckpt

# load model
latest_ckpt = select_resume_state(EVAL_DIR)
model.load_state_dict(latest_ckpt["model"], strict=False)
model = model.to(device)
model.eval()

seen = set()

with torch.no_grad():
    for i in range(len(test_df)):
        row = test_df.iloc[i]
        batch = build_row_tensors(row, device)
        if not batch.get("ok", False):
            continue

        task = str(batch["task"]).strip()
        if task in seen:
            continue

        seen.add(task)

        q = resolve_text_for_prompt(batch["qid"])
        gold = resolve_text_for_gold(task, batch["tid"])

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h = enc["h_final"]

        print("\n" + "="*80)
        print("TASK:", task)
        print("-"*80)
        # print("QUERY:")
        # print(q)

        # -------- USER --------
        if task == USER_TASK:
            probs = torch.softmax(model.heads.user_head(h), dim=-1)[0]
            topv, topi = torch.topk(probs, 25)
            gold_id = int(batch["user_labels"])
            print("\nGOLD USER:", idx2user_small[gold_id])
            print("TOP-25 USERS:")
            for v, j in zip(topv.tolist(), topi.tolist()):
                print(f"  {idx2user_small[j]:<20}  score={v:.4f}")

        # -------- ITEM --------
        if task == ITEM_TASK:
            probs = torch.softmax(model.heads.item_head(h), dim=-1)[0]
            topv, topi = torch.topk(probs, 25)
            gold_id = int(batch["item_labels"])
            print("\nGOLD ITEM:", idx2item[gold_id])
            print("TOP-25 ITEMS:")
            for v, j in zip(topv.tolist(), topi.tolist()):
                print(f"  {idx2item[j]:<30}  score={v:.4f}")

        # -------- GENERATION --------
        if task in GEN_TASKS_WORD:
            tf_pred, _ = teacher_forced_decode(model, task, h, q, gold)
            ar_pred = lm_generate_from_h(model, task, h, q)

            print("\nGOLD TEXT:")
            print(gold)

            print("\nTF ARGMAX:")
            print(tf_pred)

            print("\nAR GENERATION:")
            print(ar_pred)

        # stop after all tasks covered
        if len(seen) == (2 + len(GEN_TASKS_WORD)):
            break


In [ ]:
# =========================================================
# CELL A — Define what to finetune (UNFREEZE PLAN)
# Goal:
#   - Finetune: encoder, item_head, user_head, rwkv, bridge
#   - Finetune: last 3 transformer blocks of GPT (LM)
#   - Freeze: the rest of GPT
# Notes:
#   - This cell is safe even if some modules don't exist; it will warn you.
# =========================================================

import torch
import torch.nn as nn

def _set_requires_grad(module, flag: bool):
    if module is None:
        return
    for p in module.parameters():
        p.requires_grad = flag

def _count_params(module):
    if module is None:
        return 0
    return sum(p.numel() for p in module.parameters())

def _count_trainable(module):
    if module is None:
        return 0
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

def configure_full_finetune(model, lm_last_n_layers=3, verbose=True):
    # ---- default: freeze everything ----
    _set_requires_grad(model, False)

    # ---- unfreeze requested modules ----
    # encoder
    if hasattr(model, "encoder"):
        _set_requires_grad(model.encoder, True)
    else:
        print("[WARN] model.encoder not found")

    # heads
    if hasattr(model, "heads"):
        if hasattr(model.heads, "item_head"):
            _set_requires_grad(model.heads.item_head, True)
        else:
            print("[WARN] model.heads.item_head not found")
        if hasattr(model.heads, "user_head"):
            _set_requires_grad(model.heads.user_head, True)
        else:
            print("[WARN] model.heads.user_head not found")
    else:
        print("[WARN] model.heads not found")

    # rwkv (if you have it as a submodule)
    if hasattr(model, "rwkv"):
        _set_requires_grad(model.rwkv, True)
    else:
        print("[WARN] model.rwkv not found (skip)")

    # bridge / hint projector (common names)
    bridge_names = ["enc_bridge", "enc_bridge_proj", "bridge", "hint_bridge", "enc_to_lm"]
    found_bridge = False
    for nm in bridge_names:
        if hasattr(model, nm):
            _set_requires_grad(getattr(model, nm), True)
            found_bridge = True
    if not found_bridge:
        # if your bridge is implemented as functions only, nothing to unfreeze here
        print("[WARN] No bridge module found by common names; if your bridge has parameters, expose it as nn.Module.")

    # ---- GPT / LM: freeze all, then unfreeze last N blocks ----
    if hasattr(model, "lm"):
        lm = model.lm
        _set_requires_grad(lm, False)

        # GPT-like: lm.transformer.h (HF GPT2), or lm.model.layers (LLaMA), etc.
        # We will try multiple common layouts.
        blocks = None

        if hasattr(lm, "transformer") and hasattr(lm.transformer, "h"):
            blocks = lm.transformer.h
        elif hasattr(lm, "model") and hasattr(lm.model, "layers"):
            blocks = lm.model.layers
        elif hasattr(lm, "gpt_neox") and hasattr(lm.gpt_neox, "layers"):
            blocks = lm.gpt_neox.layers
        elif hasattr(lm, "transformer") and hasattr(lm.transformer, "layers"):
            blocks = lm.transformer.layers

        if blocks is None:
            print("[WARN] Could not locate LM blocks; leaving LM frozen.")
        else:
            n = len(blocks)
            last_n = int(min(max(lm_last_n_layers, 0), n))
            for b in blocks[n - last_n:]:
                _set_requires_grad(b, True)

            # also unfreeze final layer norms / lm_head if you want stability
            # (Optional but usually helpful when unfreezing last blocks)
            for name in ["ln_f", "norm", "final_layernorm", "ln_final"]:
                if hasattr(lm, name):
                    _set_requires_grad(getattr(lm, name), True)
            if hasattr(lm, "lm_head"):
                _set_requires_grad(lm.lm_head, True)

            if verbose:
                print(f"[LM] Unfroze last {last_n}/{n} transformer blocks (+ final norms/head if present).")
    else:
        print("[WARN] model.lm not found")

    if verbose:
        total = _count_params(model)
        trainable = _count_trainable(model)
        print("[PARAMS] total:", total, "| trainable:", trainable)

    return model

# Apply
model = configure_full_finetune(model, lm_last_n_layers=3, verbose=True)
